In [39]:
import json
import re
import os
from pathlib import Path
import pandas as pd

RESULT_FOLDER_PATH = "/Users/lockewang/FIG/WebDomainRandomizer/results_baseline"

# Load all data from uitars_predictions and uitars_metrics

In [40]:
import pandas as pd

def load_metrics_dict(metrics_file):
    """Load all metrics from uitars_metrics.jsonl into a dictionary keyed by (episode, step_index)"""
    metrics_dict = {}
    if not metrics_file.exists():
        return metrics_dict
    
    try:
        with open(metrics_file, 'r') as f:
            for line in f:
                data = json.loads(line)
                episode = data.get('episode')
                step_index = data.get('step_index')
                if episode is not None and step_index is not None:
                    key = (episode, step_index)
                    metrics_dict[key] = data.get('metrics', {})
    except Exception as e:
        print(f"Warning: Error reading metrics file {metrics_file}: {e}")
    
    return metrics_dict

def load_data_from_results_folder(result_folder_path):
    """Load all data from uitars_predictions jsonl files and match with metrics"""
    rows = []
    result_path = Path(result_folder_path)
    
    for model_config_folder in result_path.iterdir():
        if not model_config_folder.is_dir():
            continue
        if model_config_folder.name == 'locke_qwen_thought_relational_query_2':
            continue

        print(f"Processing model config folder: {model_config_folder.name}")
        
        for run_folder in model_config_folder.iterdir():
            if not run_folder.is_dir():
                continue
            
            if not run_folder.name.startswith('run'):
                continue

            model = model_config_folder.name.split('_')[1]
            reasoning_type = "no_reasoning" if "NO" in model_config_folder.name else "reasoning"
            query_type = "direct_query" if "direct_query" in model_config_folder.name else "relational_query"
            test_split = run_folder.name.split('_')[4]
            variant = "_".join(run_folder.name.split('_')[5:])
            
            predictions_folder = run_folder / "uitars_predictions"
            metrics_file = run_folder / "uitars_metrics.jsonl"
            
            if not predictions_folder.exists():
                continue
            
            # Load all metrics for this run_folder into a dictionary
            metrics_dict = load_metrics_dict(metrics_file)
            
            # Process each task_id jsonl file in uitars_predictions
            for prediction_file in predictions_folder.glob("*.jsonl"):
                task_id = prediction_file.stem  # Get task_id from filename (without .jsonl extension)
                
                try:
                    with open(prediction_file, 'r') as f:
                        for line in f:
                            data = json.loads(line)

                            episode = data.get('episode', task_id)
                            step_index = data['step_index']
                            instruction = data['instruction']
                            prediction = data['prediction']

                            # Get ground_truth_bbox from metadata
                            metadata = data.get('metadata', {})
                            gt_bbox = metadata.get('bounding_box')

                            # Get metrics from metrics_dict
                            step_metrics = metrics_dict.get((episode, step_index), {})

                            img_name = data['metadata']['screenshot'].split('/')[-1].lower()
                            image_filepath = os.path.join('~/FIG/WebDomainRandomizer/test_splits', run_folder.name, episode, 'screenshots', img_name)
                            row = {
                                'model': model,
                                'reasoning_type': reasoning_type,
                                'query_type': query_type,
                                'test_split': test_split,
                                'variant': variant,
                                'task_id': episode,
                                'step_index': step_index,
                                'instruction': instruction,
                                'gt_bbox': gt_bbox,
                                'prediction': prediction,
                                'screenshot': image_filepath,
                                'action_str_em': step_metrics.get('action_str_em'),
                                'hit_box_accuracy': step_metrics.get('hit_box_accuracy'),
                                'bbox_center_mse': step_metrics.get('bbox_center_mse')
                            }
                            
                            rows.append(row)
                            
                except Exception as e:
                    print(f"Warning: Error reading prediction file {prediction_file}: {e}")
                    continue
    
    return pd.DataFrame(rows)

# Load all data
df = load_data_from_results_folder(RESULT_FOLDER_PATH)

print(f"Total rows loaded: {len(df)}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()


Processing model config folder: locke_qwen_NO_thought_relational_query
Processing model config folder: locke_qwen_thought_relational_query
Processing model config folder: locke_gta1_NO_thought_direct_query
Processing model config folder: locke_uitars_NO_thought_relational_query
Processing model config folder: locke_uitars_thought_direct_query
Processing model config folder: locke_uitars_thought_relational_query
Processing model config folder: locke_uitars_NO_thought_direct_query
Processing model config folder: locke_qwen_thought_direct_query
Processing model config folder: locke_gta1_NO_thought_relational_query
Processing model config folder: locke_qwen_NO_thought_direct_query
Total rows loaded: 11835

Columns: ['model', 'reasoning_type', 'query_type', 'test_split', 'variant', 'task_id', 'step_index', 'instruction', 'gt_bbox', 'prediction', 'screenshot', 'action_str_em', 'hit_box_accuracy', 'bbox_center_mse']

First few rows:


,model,reasoning_type,query_type,test_split,variant,task_id,step_index,instruction,gt_bbox,prediction,screenshot,action_str_em,hit_box_accuracy,bbox_center_mse
0,qwen,no_reasoning,relational_query,task,text_shrink,8dcf6423-262a-439b-9ee7-279a920468fa,0,Click on the button to the left of 'EXPERIENCE',"[370, 61, 111.265625, 55]","Action: click(start_box='(553,86)')",~/FIG/WebDomainRandomizer/test_splits/run_20251113_005227_test_task_text_shrink/8dcf6423-262a-439b-9ee7-279a920468fa/screenshots/step_1_click.png,1.0,0.0,8114.325226
1,qwen,no_reasoning,relational_query,task,text_shrink,e6f6e6c8-f1e6-42bb-a3af-696ed8de571b,0,Click on the link above 'Flight status',"[607.046875, 174.390625, 65.390625, 12]","Action: click(start_box='(640,211)')",~/FIG/WebDomainRandomizer/test_splits/run_20251113_005227_test_task_text_shrink/e6f6e6c8-f1e6-42bb-a3af-696ed8de571b/screenshots/step_0_click.png,1.0,0.0,468.500153
2,qwen,no_reasoning,relational_query,task,text_shrink,400c291f-6a0c-46fb-874e-d5c174fdedfc,0,Click on the button below 'Infant +' button,"[895.5, 533.5, 32.5, 13]","Action: click(start_box='(834,500)')",~/FIG/WebDomainRandomizer/test_splits/run_20251113_005227_test_task_text_shrink/400c291f-6a0c-46fb-874e-d5c174fdedfc/screenshots/step_2_click.png,1.0,0.0,3822.531250
3,qwen,no_reasoning,relational_query,task,text_shrink,5b888855-b921-4c61-8f79-73902ee0eafa,0,Type the textbox above '* PIN Number',"[448, 461.953125, 380, 36.890625]","Action: click(start_box='(641,563)')",~/FIG/WebDomainRandomizer/test_splits/run_20251113_005227_test_task_text_shrink/5b888855-b921-4c61-8f79-73902ee0eafa/screenshots/step_4_type.png,0.0,0.0,3416.009064
4,qwen,no_reasoning,relational_query,task,text_shrink,9ceab2a3-7919-4f15-871a-21638fd93b24,0,"Select '11:00 AM' in the select to the left of 'Drop-off date Mon, Apr 3'","[1036.40625, 202, 112, 62]","Action: click(start_box='(1356,247)')",~/FIG/WebDomainRandomizer/test_splits/run_20251113_005227_test_task_text_shrink/9ceab2a3-7919-4f15-871a-21638fd93b24/screenshots/step_0_select.png,0.0,0.0,34838.832520


# Fix nan instruction

In [41]:
nan_instruction = df['instruction'].apply(lambda x: str(x).split(' ')[0].lower() if pd.notna(x) else '') == 'nan'
print(f"Number of rows with 'nan' instruction: {nan_instruction.sum()}")
print(f"\nAll rows with 'nan' instruction:")
display(df[nan_instruction].head(5))


nan_gt_bbox = df['gt_bbox'].apply(lambda x: x == 'nan')
print(f"Number of rows with 'nan' gt_bbox: {nan_gt_bbox.sum()}")
print(f"\nAll rows with 'nan' gt_bbox:")
display(df[nan_gt_bbox].head(5))


Number of rows with 'nan' instruction: 20

All rows with 'nan' instruction:


,model,reasoning_type,query_type,test_split,variant,task_id,step_index,instruction,gt_bbox,prediction,screenshot,action_str_em,hit_box_accuracy,bbox_center_mse
508,qwen,no_reasoning,relational_query,website,style,f5b9e5b9-1992-4444-b389-680912e50fe6,0,nan,"[31, 220.09375, 440.5, 48]","Action: click(start_box='(67,423)')",~/FIG/WebDomainRandomizer/test_splits/run_20251112_005341_test_website_style/f5b9e5b9-1992-4444-b389-680912e50fe6/screenshots/step_1_type.png,0.0,0.0,32977.754395
689,qwen,no_reasoning,relational_query,website,text_shrink,f5b9e5b9-1992-4444-b389-680912e50fe6,0,nan,"[31, 213.09375, 440.5, 48]","Action: click(start_box='(253,240)')",~/FIG/WebDomainRandomizer/test_splits/run_20251113_005230_test_website_text_shrink/f5b9e5b9-1992-4444-b389-680912e50fe6/screenshots/step_1_type.png,0.0,1.0,5.754395
741,qwen,no_reasoning,relational_query,website,precision,f5b9e5b9-1992-4444-b389-680912e50fe6,0,nan,"[281.6953125, 149.16561889648438, 322.3499755859375, 33.600006103515625]","Action: click(start_box='(445,340)')",~/FIG/WebDomainRandomizer/test_splits/run_20251112_005233_test_website_precision/f5b9e5b9-1992-4444-b389-680912e50fe6/screenshots/step_1_type.png,0.0,0.0,15146.250182
792,qwen,no_reasoning,relational_query,website,original,f5b9e5b9-1992-4444-b389-680912e50fe6,0,nan,"[31, 213.09375, 440.5, 48]","Action: click(start_box='(253,240)')",~/FIG/WebDomainRandomizer/test_splits/run_20251112_005125_test_website_original/f5b9e5b9-1992-4444-b389-680912e50fe6/screenshots/step_1_type.png,0.0,1.0,5.754395
1692,qwen,reasoning,relational_query,website,style,f5b9e5b9-1992-4444-b389-680912e50fe6,0,nan,"[31, 220.09375, 440.5, 48]","Thought: The goal is to explore job listings within a specific area. The current screen shows job results with a location filter set to ""20001"" and ""25 miles."" To view more jobs, I should scroll down to see additional listings.\n\nAction: scroll(start_box='Box(498,167,1919,1087)')",~/FIG/WebDomainRandomizer/test_splits/run_20251112_005341_test_website_style/f5b9e5b9-1992-4444-b389-680912e50fe6/screenshots/step_1_type.png,NaN,NaN,NaN


Number of rows with 'nan' gt_bbox: 0

All rows with 'nan' gt_bbox:


,model,reasoning_type,query_type,test_split,variant,task_id,step_index,instruction,gt_bbox,prediction,screenshot,action_str_em,hit_box_accuracy,bbox_center_mse


In [42]:
test_data_folder = "/Users/lockewang/FIG/WebDomainRandomizer/test_splits"
run_folders = list(Path(test_data_folder).iterdir())

def find_instruction_with_nan(row):
    test_split = row['test_split']
    variant = row['variant']
    task_id = row['task_id']
    step_index = row['step_index']

    for run_folder in run_folders:
        if test_split in run_folder.name and variant in run_folder.name:
            json_file = run_folder / task_id / "trajectory.json"
            with open(json_file, 'r') as f:
                json_data = json.load(f)
                return json_data[step_index]['step_instruction']

    raise ValueError(f"No instruction found for task_id: {task_id}, step_index: {step_index}")

df['instruction'] = df.apply(lambda x: find_instruction_with_nan(x) if x['instruction'] == 'nan' else x['instruction'], axis=1)


In [43]:
# Verify fix

nan_instruction = df['instruction'].apply(lambda x: str(x).split(' ')[0].lower() if pd.notna(x) else '') == 'nan'

len(df[nan_instruction])
df[nan_instruction].head(5)

,model,reasoning_type,query_type,test_split,variant,task_id,step_index,instruction,gt_bbox,prediction,screenshot,action_str_em,hit_box_accuracy,bbox_center_mse


# For each model_config, check how many types of predictions are there

In [44]:
def categorize_prediction_pattern(text):
    """Categorize prediction text into pattern types"""
    if pd.isna(text):
        return 'missing'
    
    text_str = str(text)
    
    types = []

    # Check if contains 'Thought' (case-insensitive)
    if 'Thought' in text_str or 'thought' in text_str:
        types.append('has_thought')
    else:
        types.append('no_thought')
    
    # Check if contains 'tool_call' (case-insensitive)
    if 'tool_call' in text_str or 'toolCall' in text_str or 'tool-call' in text_str:
        types.append('contains_tool_call')
    
    # Check if has exactly 4 numbers (could be bbox format like [x1, y1, x2, y2] or (x1, y1, x2, y2))
    numbers = re.findall(r'\d+', text_str)
    if len(numbers) == 4:
        types.append('4_numbers')
    elif len(numbers) == 0:
        types.append('no_numbers')

    # Extract all actions after "Action:" - handle multiple actions separated by commas
    action_section_match = re.search(r'Action:\s*(.+?)(?:\n|$)', text_str, re.IGNORECASE | re.DOTALL)
    if action_section_match:
        action_section = action_section_match.group(1).strip()
        
        # Parse multiple actions (they're separated by commas, but we need to be careful with nested parentheses)
        # Simple approach: find all action calls like action_name(...)
        action_pattern = r'\b(click|scroll|type|select|wait|finished)\s*\('
        found_actions = re.findall(action_pattern, action_section, re.IGNORECASE)
        
        # Add all found action types
        for action in found_actions:
            action_lower = action.lower()
            if action_lower not in [t.lower() for t in types]:
                types.append(action_lower)
    
    # Everything else is unexpected
    if len(types) == 0:
        types.append('other_unexpected')

    return ' + '.join(types)

In [45]:
pd.set_option('display.max_rows', None)

df['prediction_type'] = df['prediction'].apply(categorize_prediction_pattern)

other_unexpected_count = df[df['prediction_type'] == 'other_unexpected'].sum()
other_unexpected_count


model                 0
reasoning_type        0
query_type            0
test_split            0
variant               0
task_id               0
step_index            0
instruction           0
gt_bbox               0
prediction            0
screenshot            0
action_str_em       0.0
hit_box_accuracy    0.0
bbox_center_mse     0.0
prediction_type       0
dtype: object

In [46]:
# filter out rows with expected prediction types
# reasoning_type is reasoning, prediction_type should only have contains_thought and other actions like select, scroll, type, etc.
# reasoning_type is no_reasoning, prediction_type should only have simple_click and other actions like select, scroll, type, etc.


# Define allowed actions for each reasoning type
reasoning_allowed = ['has_thought', 'click', 'scroll', 'type', 'select', 'wait', 'finished']
no_reasoning_allowed = ['no_thought', 'click', 'scroll', 'type', 'select', 'wait', 'finished']


has_valid_reasoning_actions = df['prediction_type'].apply(lambda x: all(action in reasoning_allowed for action in x.split(' + ')))
has_valid_no_reasoning_actions = df['prediction_type'].apply(lambda x: all(action in no_reasoning_allowed for action in x.split(' + ')))

# Combine conditions
expected_prediction_filter = (
    ((df['reasoning_type'] == 'reasoning') & has_valid_reasoning_actions) |
    ((df['reasoning_type'] == 'no_reasoning') & has_valid_no_reasoning_actions)
)

filtered_df = df[~expected_prediction_filter]
expected_df = df[expected_prediction_filter]

print(f'Filtered out {len(filtered_df)} / {len(df)} rows')
display(filtered_df.groupby(['model', 'reasoning_type', 'prediction_type'])['prediction'].count())
print("="*80)
display(expected_df.groupby(['model', 'reasoning_type', 'prediction_type'])['prediction'].count())
print("="*80)

Filtered out 1337 / 11835 rows


model   reasoning_type  prediction_type                             
qwen    no_reasoning    no_thought + contains_tool_call                   2
                        no_thought + contains_tool_call + no_numbers      1
        reasoning       has_thought + 4_numbers + click                 896
                        has_thought + 4_numbers + scroll                112
                        has_thought + 4_numbers + scroll + wait           2
                        has_thought + 4_numbers + wait                    1
                        has_thought + no_numbers + click                  1
                        has_thought + no_numbers + finished              11
                        has_thought + no_numbers + type                  26
                        has_thought + no_numbers + wait                   3
uitars  reasoning       has_thought + 4_numbers + click                 122
                        has_thought + 4_numbers + scroll                  4
                   

model   reasoning_type  prediction_type            
gta1    no_reasoning    no_thought + click             2368
qwen    no_reasoning    no_thought + click             2365
        reasoning       has_thought + click            1050
                        has_thought + finished            2
                        has_thought + scroll            237
                        has_thought + scroll + wait      16
                        has_thought + type                7
                        has_thought + wait                1
uitars  no_reasoning    no_thought + click             2234
                        no_thought + scroll             134
        reasoning       has_thought + click            1937
                        has_thought + finished            3
                        has_thought + scroll            117
                        has_thought + wait               27
Name: prediction, dtype: int64

## Unexpected predictions
Only Qwen has all 3 malformed predictions, and UITARS has some 4D coordinates cases
1. has <tool_call>...scroll....pixel: -500... or <tool_call>...type...content: PUNT...
2. has 4D coordinates instead 2D
3. had 2 actions, the only case is ["scroll", "wait"]

Side note, 
- our original "predicted_actions" field sometimes has 2 actions like 'click', 'type'. But this is from our parsing logic. The 'prediction' string has 'type' in its 'Thought' but only has 'click' in 'Action'
- saved 'prediction' is not raw prediction, it's formatted prediction. The regex patterns used miss coordinates formatted with (x, y) instead of [x, y] for Qwen. However, we can count (x, y) just as malformed like other cases (e.g., 'scroll... pixel: -500')


In [47]:
for model in filtered_df['model'].unique():
    result = filtered_df[filtered_df['model'] == model].groupby(['model', 'reasoning_type', 'prediction_type'])['prediction'].count()
    styled = (result.reset_index(name='count')
              .style
              .background_gradient(subset='count')
              .format({'count': '{:,}'}))
    print("="*80)
    print(model)
    display(styled)
    print("="*80)


qwen


,model,reasoning_type,prediction_type,count
0,qwen,no_reasoning,no_thought + contains_tool_call,2
1,qwen,no_reasoning,no_thought + contains_tool_call + no_numbers,1
2,qwen,reasoning,has_thought + 4_numbers + click,896
3,qwen,reasoning,has_thought + 4_numbers + scroll,112
4,qwen,reasoning,has_thought + 4_numbers + scroll + wait,2
5,qwen,reasoning,has_thought + 4_numbers + wait,1
6,qwen,reasoning,has_thought + no_numbers + click,1
7,qwen,reasoning,has_thought + no_numbers + finished,11
8,qwen,reasoning,has_thought + no_numbers + type,26
9,qwen,reasoning,has_thought + no_numbers + wait,3


uitars


,model,reasoning_type,prediction_type,count
0,uitars,reasoning,has_thought + 4_numbers + click,122
1,uitars,reasoning,has_thought + 4_numbers + scroll,4
2,uitars,reasoning,has_thought + 4_numbers + wait,4
3,uitars,reasoning,has_thought + no_numbers + finished,10
4,uitars,reasoning,has_thought + no_numbers + type,3
5,uitars,reasoning,has_thought + no_numbers + wait,139


In [48]:
for model in filtered_df['model'].unique():
    result = filtered_df[filtered_df['model'] == model].groupby(['model', 'reasoning_type', 'query_type', 'prediction_type'])['prediction'].count()
    styled = (result.reset_index(name='count')
              .style
              .background_gradient(subset='count')
              .format({'count': '{:,}'}))
    print("="*80)
    print(model)
    display(styled)
    print("="*80)

qwen


,model,reasoning_type,query_type,prediction_type,count
0,qwen,no_reasoning,direct_query,no_thought + contains_tool_call + no_numbers,1
1,qwen,no_reasoning,relational_query,no_thought + contains_tool_call,2
2,qwen,reasoning,direct_query,has_thought + 4_numbers + click,435
3,qwen,reasoning,direct_query,has_thought + 4_numbers + scroll,70
4,qwen,reasoning,direct_query,has_thought + 4_numbers + scroll + wait,1
5,qwen,reasoning,direct_query,has_thought + 4_numbers + wait,1
6,qwen,reasoning,direct_query,has_thought + no_numbers + click,1
7,qwen,reasoning,direct_query,has_thought + no_numbers + finished,5
8,qwen,reasoning,direct_query,has_thought + no_numbers + type,20
9,qwen,reasoning,relational_query,has_thought + 4_numbers + click,461


uitars


,model,reasoning_type,query_type,prediction_type,count
0,uitars,reasoning,direct_query,has_thought + 4_numbers + click,62
1,uitars,reasoning,direct_query,has_thought + 4_numbers + scroll,3
2,uitars,reasoning,direct_query,has_thought + 4_numbers + wait,3
3,uitars,reasoning,direct_query,has_thought + no_numbers + finished,7
4,uitars,reasoning,direct_query,has_thought + no_numbers + type,2
5,uitars,reasoning,direct_query,has_thought + no_numbers + wait,59
6,uitars,reasoning,relational_query,has_thought + 4_numbers + click,60
7,uitars,reasoning,relational_query,has_thought + 4_numbers + scroll,1
8,uitars,reasoning,relational_query,has_thought + 4_numbers + wait,1
9,uitars,reasoning,relational_query,has_thought + no_numbers + finished,3


## Expected predictions

In [49]:
for model in expected_df['model'].unique():
    result = expected_df[expected_df['model'] == model].groupby(['model', 'reasoning_type', 'query_type', 'prediction_type'])['prediction'].count()
    styled = (result.reset_index(name='count')
              .style
              .background_gradient(subset='count')
              .format({'count': '{:,}'}))
    print("="*80)
    print(model)
    display(styled)
    print("="*80)

qwen


,model,reasoning_type,query_type,prediction_type,count
0,qwen,no_reasoning,direct_query,no_thought + click,"1,183"
1,qwen,no_reasoning,relational_query,no_thought + click,"1,182"
2,qwen,reasoning,direct_query,has_thought + click,491
3,qwen,reasoning,direct_query,has_thought + scroll,150
4,qwen,reasoning,direct_query,has_thought + scroll + wait,6
5,qwen,reasoning,direct_query,has_thought + type,3
6,qwen,reasoning,direct_query,has_thought + wait,1
7,qwen,reasoning,relational_query,has_thought + click,559
8,qwen,reasoning,relational_query,has_thought + finished,2
9,qwen,reasoning,relational_query,has_thought + scroll,87


gta1


,model,reasoning_type,query_type,prediction_type,count
0,gta1,no_reasoning,direct_query,no_thought + click,"1,184"
1,gta1,no_reasoning,relational_query,no_thought + click,"1,184"


uitars


,model,reasoning_type,query_type,prediction_type,count
0,uitars,no_reasoning,direct_query,no_thought + click,"1,148"
1,uitars,no_reasoning,direct_query,no_thought + scroll,36
2,uitars,no_reasoning,relational_query,no_thought + click,"1,086"
3,uitars,no_reasoning,relational_query,no_thought + scroll,98
4,uitars,reasoning,direct_query,has_thought + click,983
5,uitars,reasoning,direct_query,has_thought + scroll,59
6,uitars,reasoning,direct_query,has_thought + wait,6
7,uitars,reasoning,relational_query,has_thought + click,954
8,uitars,reasoning,relational_query,has_thought + finished,3
9,uitars,reasoning,relational_query,has_thought + scroll,58


# Parsing actions & coordinates from saved 'prediction' field in the df from the jsonl files

In [50]:
df.head()

,model,reasoning_type,query_type,test_split,variant,task_id,step_index,instruction,gt_bbox,prediction,screenshot,action_str_em,hit_box_accuracy,bbox_center_mse,prediction_type
0,qwen,no_reasoning,relational_query,task,text_shrink,8dcf6423-262a-439b-9ee7-279a920468fa,0,Click on the button to the left of 'EXPERIENCE',"[370, 61, 111.265625, 55]","Action: click(start_box='(553,86)')",~/FIG/WebDomainRandomizer/test_splits/run_20251113_005227_test_task_text_shrink/8dcf6423-262a-439b-9ee7-279a920468fa/screenshots/step_1_click.png,1.0,0.0,8114.325226,no_thought + click
1,qwen,no_reasoning,relational_query,task,text_shrink,e6f6e6c8-f1e6-42bb-a3af-696ed8de571b,0,Click on the link above 'Flight status',"[607.046875, 174.390625, 65.390625, 12]","Action: click(start_box='(640,211)')",~/FIG/WebDomainRandomizer/test_splits/run_20251113_005227_test_task_text_shrink/e6f6e6c8-f1e6-42bb-a3af-696ed8de571b/screenshots/step_0_click.png,1.0,0.0,468.500153,no_thought + click
2,qwen,no_reasoning,relational_query,task,text_shrink,400c291f-6a0c-46fb-874e-d5c174fdedfc,0,Click on the button below 'Infant +' button,"[895.5, 533.5, 32.5, 13]","Action: click(start_box='(834,500)')",~/FIG/WebDomainRandomizer/test_splits/run_20251113_005227_test_task_text_shrink/400c291f-6a0c-46fb-874e-d5c174fdedfc/screenshots/step_2_click.png,1.0,0.0,3822.531250,no_thought + click
3,qwen,no_reasoning,relational_query,task,text_shrink,5b888855-b921-4c61-8f79-73902ee0eafa,0,Type the textbox above '* PIN Number',"[448, 461.953125, 380, 36.890625]","Action: click(start_box='(641,563)')",~/FIG/WebDomainRandomizer/test_splits/run_20251113_005227_test_task_text_shrink/5b888855-b921-4c61-8f79-73902ee0eafa/screenshots/step_4_type.png,0.0,0.0,3416.009064,no_thought + click
4,qwen,no_reasoning,relational_query,task,text_shrink,9ceab2a3-7919-4f15-871a-21638fd93b24,0,"Select '11:00 AM' in the select to the left of 'Drop-off date Mon, Apr 3'","[1036.40625, 202, 112, 62]","Action: click(start_box='(1356,247)')",~/FIG/WebDomainRandomizer/test_splits/run_20251113_005227_test_task_text_shrink/9ceab2a3-7919-4f15-871a-21638fd93b24/screenshots/step_0_select.png,0.0,0.0,34838.832520,no_thought + click


In [51]:
# parsing prediction from the df

def parse_actions(prediction_str):
    """
    Extract all actions from a prediction string.
    Handles: "Action: click(...)", JSON in tool_call tags, and malformed cases.
    Returns a list of dicts with 'action_name' and 'params'.
    """
    if pd.isna(prediction_str):
        return []
    
    text_str = str(prediction_str)
    actions = []
    action_types = ['click', 'scroll', 'type', 'select', 'wait', 'finished']
    
    # Helper: Find balanced braces/parentheses
    def find_balanced(text, start_pos, open_char='(', close_char=')'):
        depth = 0
        for i in range(start_pos, len(text)):
            if text[i] == open_char:
                depth += 1
            elif text[i] == close_char:
                depth -= 1
                if depth == 0:
                    return i
        return -1
    
    # Helper: Parse coordinates from various formats
    def parse_coordinates(params_str):
        """Extract coordinates from params string. Returns dict with 'coords_2d', 'coords_4d', or None"""
        if not params_str:
            return None
        
        # Try to find numbers in the params
        numbers = re.findall(r'-?\d+\.?\d*', str(params_str))
        if not numbers:
            return None
        
        # Convert to floats/ints
        try:
            coords = [float(n) if '.' in n else int(n) for n in numbers]
        except ValueError:
            return None
        
        result = {}
        
        # Check for 2D coordinates (2 numbers)
        if len(coords) >= 2:
            result['coords_2d'] = coords[:2]
        
        # Check for 4D coordinates (4 numbers - bounding box)
        if len(coords) >= 4:
            result['coords_4d'] = coords[:4]
        
        # If we have more than 4 numbers, store them as 'coords_extra'
        if len(coords) > 4:
            result['coords_extra'] = coords
        
        return result if result else None
    
    # Helper: Extract params from JSON arguments based on action type
    def extract_json_params(action_name, args):
        """Extract relevant parameter from JSON arguments"""
        import json
        param_map = {
            'scroll': ['pixels', 'direction'],
            'type': ['text', 'content'],
            'click': ['start_box', 'coordinates']
        }
        for key in param_map.get(action_name, []):
            if key in args:
                return str(args[key])
        # Fallback: return all args except 'action'
        params = {k: v for k, v in args.items() if k != 'action'}
        return json.dumps(params, separators=(',', ':')) if params else None
    
    # Helper: Parse JSON and extract action
    def parse_json_action(json_str):
        """Parse JSON and return action dict if valid"""
        try:
            import json
            data = json.loads(json_str)
            if not isinstance(data, dict):
                return None
            
            # Find action name
            action_name = None
            if 'name' in data and data['name'] in action_types:
                action_name = data['name'].lower()
            elif 'arguments' in data and isinstance(data['arguments'], dict):
                if 'action' in data['arguments'] and data['arguments']['action'] in action_types:
                    action_name = data['arguments']['action'].lower()
            
            if not action_name:
                return None
            
            # Extract params
            params = None
            if 'arguments' in data and isinstance(data['arguments'], dict):
                params = extract_json_params(action_name, data['arguments'])
            
            action_dict = {
                'action_name': action_name,
                'params': params,
                'raw': json_str,
                'malformed': True
            }
            
            # Parse coordinates from params
            coords = parse_coordinates(params)
            if coords:
                action_dict.update(coords)
            
            return action_dict
        except (json.JSONDecodeError, ValueError):
            return None
    
    # Strategy 1: Well-formed actions after "Action:"
    action_section_match = re.search(r'Action:\s*(.+?)(?:\n|$)', text_str, re.IGNORECASE | re.DOTALL)
    if action_section_match:
        action_section = action_section_match.group(1).strip()
        for action_type in action_types:
            for match in re.finditer(rf'\b{action_type}\s*\(', action_section, re.IGNORECASE):
                start_pos = match.end() - 1
                end_pos = find_balanced(action_section, start_pos)
                if end_pos != -1:
                    params = action_section[match.end():end_pos].strip()
                    action_dict = {
                        'action_name': action_type.lower(),
                        'params': params,
                        'raw': action_section[match.start():end_pos + 1]
                    }
                    # Parse coordinates from params
                    coords = parse_coordinates(params)
                    if coords:
                        action_dict.update(coords)
                    actions.append(action_dict)
    
    # Strategy 2: JSON in tool_call tags or standalone JSON
    if len(actions) == 0:
        # Helper: Find all JSON objects starting at a position
        def find_all_json(text, start_pos=0):
            json_objects = []
            i = start_pos
            while i < len(text):
                if text[i] == '{':
                    end_pos = find_balanced(text, i, '{', '}')
                    if end_pos != -1:
                        json_objects.append(text[i:end_pos + 1])
                        i = end_pos + 1
                    else:
                        i += 1
                else:
                    i += 1
            return json_objects
        
        # Try tool_call tags first - find ALL JSON objects in tool_call sections
        tool_call_matches = list(re.finditer(r'<tool_call>', text_str, re.IGNORECASE))
        for tool_call_match in tool_call_matches:
            json_objects = find_all_json(text_str, tool_call_match.end())
            for json_str in json_objects:
                action = parse_json_action(json_str)
                if action:
                    actions.append(action)
        
        # Try finding any JSON with "action" field (if no tool_call found)
        if len(actions) == 0:
            json_objects = find_all_json(text_str)
            for json_str in json_objects:
                if '"action"' in json_str or "'action'" in json_str:
                    action = parse_json_action(json_str)
                    if action:
                        actions.append(action)
    
    # Strategy 3: Fallback - find action keywords, but EXCLUDE "Thought:" sections
    if len(actions) == 0:
        # Remove "Thought:" sections to avoid false matches
        # Find all "Thought:" sections and remove them
        search_text = text_str
        thought_pattern = r'Thought:\s*[^\n]*(?:\n|$)'
        search_text = re.sub(thought_pattern, '', search_text, flags=re.IGNORECASE | re.MULTILINE)
        
        # Search for action keywords in the filtered text (excluding Thought sections)
        for action_type in action_types:
            match = re.search(rf'\b{action_type}\b', search_text, re.IGNORECASE)
            if match:
                # Try to find coordinates near the action keyword (within 100 chars)
                start = max(0, match.start() - 50)
                end = min(len(search_text), match.end() + 50)
                context = search_text[start:end]
                
                action_dict = {
                    'action_name': action_type.lower(),
                    'params': None,
                    'raw': action_type,
                    'malformed': True
                }
                
                # Parse coordinates from context around the action
                coords = parse_coordinates(context)
                if coords:
                    action_dict.update(coords)
                
                actions.append(action_dict)
    
    return actions

# Test the function and apply to dataframe
df['parsed_actions'] = df['prediction'].apply(parse_actions)

# Also create a column with just action names for easier analysis
df['action_names'] = df['parsed_actions'].apply(lambda x: [a['action_name'] for a in x] if x else [])
df['action_count'] = df['parsed_actions'].apply(len)

# Extract coordinates from parsed actions for easier metric calculation
# Take coordinates from the first action (if any)
def extract_coords_from_actions(actions):
    """Extract coordinates from the first action"""
    if not actions or len(actions) == 0:
        return {
            'coords_2d': None,
            'coords_4d': None,
            'coords_extra': None
        }
    
    first_action = actions[0]
    return {
        'coords_2d': first_action.get('coords_2d'),
        'coords_4d': first_action.get('coords_4d'),
        'coords_extra': first_action.get('coords_extra')
    }

coords_df = pd.DataFrame(df['parsed_actions'].apply(extract_coords_from_actions).tolist())
df['coords_2d'] = coords_df['coords_2d']
df['coords_4d'] = coords_df['coords_4d']
df['coords_extra'] = coords_df['coords_extra']

# show full width
pd.set_option('display.max_colwidth', None)

# Filter for rows with 0 actions (malformed/missing) OR more than 1 action (multiple actions)
more_than_1_action = (df['prediction'].str.contains('tool_call')) | (df['action_count'] > 1) | (df['action_count'] == 0)

print(f"Sample parsed actions:")
print(f"Rows with 0 or more than 1 action: {more_than_1_action.sum()}")
print(f"Rows with exactly 1 action: {(df['action_count'] == 1).sum()}")
print(f"\nRows with coords_2d: {df['coords_2d'].notna().sum()}")
print(f"Rows with coords_4d: {df['coords_4d'].notna().sum()}")
display(df[more_than_1_action][['model','instruction', 'prediction', 'parsed_actions', 'coords_2d', 'coords_4d']].head())

Sample parsed actions:
Rows with 0 or more than 1 action: 21
Rows with exactly 1 action: 11817

Rows with coords_2d: 11590
Rows with coords_4d: 1257


,model,instruction,prediction,parsed_actions,coords_2d,coords_4d
230,qwen,Type 'Home Decor Inspiration' above 'Keep this board secret',"<tool_call>\n{""name"": ""computer_use"", ""arguments"": {""action"": ""scroll"", ""pixels"": -505}}\n</tool_call>","[{'action_name': 'scroll', 'params': '-505', 'raw': '{""name"": ""computer_use"", ""arguments"": {""action"": ""scroll"", ""pixels"": -505}}', 'malformed': True}]",None,None
1055,qwen,Type 'Home Decor Inspiration' above 'Keep this board secret',"<tool_call>\n{""name"": ""computer_use"", ""arguments"": {""action"": ""scroll"", ""pixels"": -505}}\n</tool_call>","[{'action_name': 'scroll', 'params': '-505', 'raw': '{""name"": ""computer_use"", ""arguments"": {""action"": ""scroll"", ""pixels"": -505}}', 'malformed': True}]",None,None
1250,qwen,Click on the button above 'Done',"Thought: The goal is to click on the button labeled ""Done"" which is likely located above the current view. Since it's not visible in the screenshot, I will scroll down to locate it.\nAction: scroll(start_box='Box(0,0)'), wait()","[{'action_name': 'scroll', 'params': 'start_box='Box(0,0)'', 'raw': 'scroll(start_box='Box(0,0)')', 'coords_2d': [0, 0]}, {'action_name': 'wait', 'params': '', 'raw': 'wait()'}]","[0, 0]",None
1572,qwen,Click on the text below 'Cheapest',"Thought: The task is to click on the text below 'Cheapest'. I need to locate the text and click on it. The text is likely to be near the bottom of the screen, so I will scroll down slightly if necessary before clicking.\n\nAction: scroll(start_box='Box(498,602,1931,1087)'), wait()","[{'action_name': 'scroll', 'params': 'start_box='Box(498,602,1931,1087)'', 'raw': 'scroll(start_box='Box(498,602,1931,1087)')', 'coords_2d': [498, 602], 'coords_4d': [498, 602, 1931, 1087]}, {'action_name': 'wait', 'params': '', 'raw': 'wait()'}]","[498, 602]","[498, 602, 1931, 1087]"
1634,qwen,Click on the link above 'Italy',"Thought: The goal is to click on the link labeled 'Italy' which is located above the current view. This will likely lead to a page specific to vinyl records from Italy. The link is not visible in the current screenshot, so I need to scroll down to find it.\n\nAction: scroll(start_box='Box(0,0)'), wait()","[{'action_name': 'scroll', 'params': 'start_box='Box(0,0)'', 'raw': 'scroll(start_box='Box(0,0)')', 'coords_2d': [0, 0]}, {'action_name': 'wait', 'params': '', 'raw': 'wait()'}]","[0, 0]",None


In [52]:
contains_type_action = df['prediction'].str.contains('type')
print(f"Rows with type action: {contains_type_action.sum()}")
# display(df[contains_type_action][['model','instruction', 'prediction', 'parsed_actions', 'action_names', 'action_count', 'coords_2d', 'coords_4d']].head(-10))
display(df[contains_type_action][['model', 'reasoning_type', 'parsed_actions', 'prediction_type', 'action_names']].groupby(['model', 'reasoning_type', 'prediction_type']).head(5))


Rows with type action: 304


,model,reasoning_type,parsed_actions,prediction_type,action_names
1187,qwen,reasoning,"[{'action_name': 'click', 'params': 'start_box='_bbox(449,546),(832,583)_'', 'raw': 'click(start_box='_bbox(449,546),(832,583)_')', 'coords_2d': [449, 546], 'coords_4d': [449, 546, 832, 583]}]",has_thought + 4_numbers + click,[click]
1197,qwen,reasoning,"[{'action_name': 'click', 'params': 'start_box='_bbox(1029,430,1491,482)_bbox'', 'raw': 'click(start_box='_bbox(1029,430,1491,482)_bbox')', 'coords_2d': [1029, 430], 'coords_4d': [1029, 430, 1491, 482]}]",has_thought + 4_numbers + click,[click]
1216,qwen,reasoning,"[{'action_name': 'click', 'params': 'start_box='_bbox(136,229),(478,407)_'', 'raw': 'click(start_box='_bbox(136,229),(478,407)_')', 'coords_2d': [136, 229], 'coords_4d': [136, 229, 478, 407]}]",has_thought + 4_numbers + click,[click]
1232,qwen,reasoning,"[{'action_name': 'click', 'params': 'start_box='(602,47)'', 'raw': 'click(start_box='(602,47)')', 'coords_2d': [602, 47]}]",has_thought + click,[click]
1246,qwen,reasoning,"[{'action_name': 'click', 'params': 'start_box='[1204,121]'', 'raw': 'click(start_box='[1204,121]')', 'coords_2d': [1204, 121]}]",has_thought + click,[click]
1260,qwen,reasoning,"[{'action_name': 'type', 'params': 'content='42701\n'', 'raw': 'type(content='42701\n')'}]",has_thought + type,[type]
1262,qwen,reasoning,"[{'action_name': 'click', 'params': 'start_box='_bbox(847,70,912,106)'', 'raw': 'click(start_box='_bbox(847,70,912,106)')', 'coords_2d': [847, 70], 'coords_4d': [847, 70, 912, 106]}]",has_thought + 4_numbers + click,[click]
1275,qwen,reasoning,"[{'action_name': 'click', 'params': 'start_box='(480,130)'', 'raw': 'click(start_box='(480,130)')', 'coords_2d': [480, 130]}]",has_thought + 4_numbers + click,[click]
1305,qwen,reasoning,"[{'action_name': 'click', 'params': 'start_box='[1142,160]'', 'raw': 'click(start_box='[1142,160]')', 'coords_2d': [1142, 160]}]",has_thought + click,[click]
1311,qwen,reasoning,"[{'action_name': 'click', 'params': 'start_box='_bbox(867,402,911,426)_'', 'raw': 'click(start_box='_bbox(867,402,911,426)_')', 'coords_2d': [867, 402], 'coords_4d': [867, 402, 911, 426]}]",has_thought + click,[click]


In [53]:
has_type = (df['prediction'].str.contains('type')) & (df['action_names'].apply(lambda x: 'type' in x))
df[has_type][['model', 'reasoning_type', 'prediction_type', 'prediction']].groupby(['model', 'reasoning_type', 'prediction_type']).value_counts()

model   reasoning_type  prediction_type                               prediction                                                                                                                                                                                                                                                                                                                                        
qwen    no_reasoning    no_thought + contains_tool_call + no_numbers  <tool_call>\n{"name": "computer_use", "arguments": {"action": "type", "text": "PUNE"}}\n</tool_call>                                                                                                                                                                                                                                                  1
        reasoning       has_thought + no_numbers + type               Thought: The goal is to type 'diabetes' into the search box. The search box is clearly visible and ready fo

# Fixing coordinates by adding correct coordinate denormalization using W_RATIO & H_RATIO & converting 4D predictions to 2D predictions

In [54]:
SMART_RESIZE = (1932, 1092)
IMAGE_SIZE = (1920, 1080)

W_RATIO = IMAGE_SIZE[0] / SMART_RESIZE[0]
H_RATIO = IMAGE_SIZE[1] / SMART_RESIZE[1]

In [55]:
df.head()

,model,reasoning_type,query_type,test_split,variant,task_id,step_index,instruction,gt_bbox,prediction,...,action_str_em,hit_box_accuracy,bbox_center_mse,prediction_type,parsed_actions,action_names,action_count,coords_2d,coords_4d,coords_extra
0,qwen,no_reasoning,relational_query,task,text_shrink,8dcf6423-262a-439b-9ee7-279a920468fa,0,Click on the button to the left of 'EXPERIENCE',"[370, 61, 111.265625, 55]","Action: click(start_box='(553,86)')",...,1.0,0.0,8114.325226,no_thought + click,"[{'action_name': 'click', 'params': 'start_box='(553,86)'', 'raw': 'click(start_box='(553,86)')', 'coords_2d': [553, 86]}]",[click],1,"[553, 86]",None,None
1,qwen,no_reasoning,relational_query,task,text_shrink,e6f6e6c8-f1e6-42bb-a3af-696ed8de571b,0,Click on the link above 'Flight status',"[607.046875, 174.390625, 65.390625, 12]","Action: click(start_box='(640,211)')",...,1.0,0.0,468.500153,no_thought + click,"[{'action_name': 'click', 'params': 'start_box='(640,211)'', 'raw': 'click(start_box='(640,211)')', 'coords_2d': [640, 211]}]",[click],1,"[640, 211]",None,None
2,qwen,no_reasoning,relational_query,task,text_shrink,400c291f-6a0c-46fb-874e-d5c174fdedfc,0,Click on the button below 'Infant +' button,"[895.5, 533.5, 32.5, 13]","Action: click(start_box='(834,500)')",...,1.0,0.0,3822.531250,no_thought + click,"[{'action_name': 'click', 'params': 'start_box='(834,500)'', 'raw': 'click(start_box='(834,500)')', 'coords_2d': [834, 500]}]",[click],1,"[834, 500]",None,None
3,qwen,no_reasoning,relational_query,task,text_shrink,5b888855-b921-4c61-8f79-73902ee0eafa,0,Type the textbox above '* PIN Number',"[448, 461.953125, 380, 36.890625]","Action: click(start_box='(641,563)')",...,0.0,0.0,3416.009064,no_thought + click,"[{'action_name': 'click', 'params': 'start_box='(641,563)'', 'raw': 'click(start_box='(641,563)')', 'coords_2d': [641, 563]}]",[click],1,"[641, 563]",None,None
4,qwen,no_reasoning,relational_query,task,text_shrink,9ceab2a3-7919-4f15-871a-21638fd93b24,0,"Select '11:00 AM' in the select to the left of 'Drop-off date Mon, Apr 3'","[1036.40625, 202, 112, 62]","Action: click(start_box='(1356,247)')",...,0.0,0.0,34838.832520,no_thought + click,"[{'action_name': 'click', 'params': 'start_box='(1356,247)'', 'raw': 'click(start_box='(1356,247)')', 'coords_2d': [1356, 247]}]",[click],1,"[1356, 247]",None,None


In [56]:
"""
1. coords_4d exists -> calculate center -> corrected_coords_2d
2. coords_4d doesn't exist -> coords_2d is corrected_coords_2d
3. coords_extra exists -> does step 1
4. coords_2d is None likely due to Type action -> save None coords_2d as corrected_coords_2d
"""

def get_center_from_qwen_coords_4d(coords_4d):
    x1, y1, x2, y2 = coords_4d
    return [(x1 + x2) / 2, (y1 + y2) / 2]

# get center from coords_4d for qwen rows with 4d coordinates
df['corrected_coords_2d'] = df.apply(lambda x: get_center_from_qwen_coords_4d(x['coords_4d']) if x['coords_4d'] is not None else x['coords_2d'], axis=1)

# denormalize coords_2d for all models
df['corrected_coords_2d_denormalized'] = df.apply(lambda x: [int(x['corrected_coords_2d'][0] * W_RATIO), int(x['corrected_coords_2d'][1] * H_RATIO)] if x['corrected_coords_2d'] is not None else x['corrected_coords_2d'], axis=1)


In [57]:
df[df['coords_4d'].notna()][['model', 'reasoning_type', 'coords_2d', 'coords_4d', 'corrected_coords_2d', 'corrected_coords_2d_denormalized']].head()

,model,reasoning_type,coords_2d,coords_4d,corrected_coords_2d,corrected_coords_2d_denormalized
1184,qwen,reasoning,"[524, 80]","[524, 80, 597, 93]","[560.5, 86.5]","[557, 85]"
1186,qwen,reasoning,"[620, 484]","[620, 484, 679, 502]","[649.5, 493.0]","[645, 487]"
1187,qwen,reasoning,"[449, 546]","[449, 546, 832, 583]","[640.5, 564.5]","[636, 558]"
1188,qwen,reasoning,"[1297, 204]","[1297, 204, 1414, 268]","[1355.5, 236.0]","[1347, 233]"
1190,qwen,reasoning,"[568, 409]","[568, 409, 1247, 481]","[907.5, 445.0]","[901, 440]"


# Fix the action str exact match, hit box accuracy, & bbox center mse, and add normalized mse by bbox diagonal length

## Calculate action str exact match

1. malformed -> 0
2. non-coordinate actions -> action type and params must match
3. normal click action with coordinates case, see if action type matches
4. for steps with 2 actions, it's only scroll, wait. and we take the first action 'scroll'

In [58]:
df[df['action_count'] != 1]['parsed_actions'].head()

1250                                                                         [{'action_name': 'scroll', 'params': 'start_box='Box(0,0)'', 'raw': 'scroll(start_box='Box(0,0)')', 'coords_2d': [0, 0]}, {'action_name': 'wait', 'params': '', 'raw': 'wait()'}]
1572    [{'action_name': 'scroll', 'params': 'start_box='Box(498,602,1931,1087)'', 'raw': 'scroll(start_box='Box(498,602,1931,1087)')', 'coords_2d': [498, 602], 'coords_4d': [498, 602, 1931, 1087]}, {'action_name': 'wait', 'params': '', 'raw': 'wait()'}]
1634                                                                         [{'action_name': 'scroll', 'params': 'start_box='Box(0,0)'', 'raw': 'scroll(start_box='Box(0,0)')', 'coords_2d': [0, 0]}, {'action_name': 'wait', 'params': '', 'raw': 'wait()'}]
1660                                                                   [{'action_name': 'scroll', 'params': 'start_box='Box(10,10)'', 'raw': 'scroll(start_box='Box(10,10)')', 'coords_2d': [10, 10]}, {'action_name': 'wait', 'params': ''

In [59]:
# We are only counting scroll as mismatch
# Type
# Click
# Select
# Scroll
# Wait
# Finished

def get_action_str_em(row):
    gt_action_type = row['instruction'].split(' ')[0].lower()
    prediction_action_name = row['parsed_actions'][0]['action_name'] # always take the first action name

    if prediction_action_name == 'type':
        action_str_em = 1 if gt_action_type == 'type' else 0
    elif prediction_action_name == 'click':
        action_str_em = 1  # click is always correct due to our parsing logic
    elif prediction_action_name == 'scroll':
        action_str_em = 1 if gt_action_type == 'scroll' else 0
    elif prediction_action_name == 'select':
        action_str_em = 1 if gt_action_type == 'select' else 0
    elif prediction_action_name == 'wait':
        action_str_em = 1 if gt_action_type == 'wait' else 0
    else:
        return 0

    return action_str_em

df['corrected_action_str_em'] = df.apply(get_action_str_em, axis=1)


## Analyze prediction action types

In [60]:
# check action str em stats
print("Value counts by group:")
display(df.groupby(['model', 'reasoning_type', 'query_type', 'variant'])['corrected_action_str_em'].value_counts())

print("\n" + "="*80)
print("Percentage of corrected_action_str_em == 1 by group:")
print("="*80)

# Calculate percentage at each grouping level
group_cols = ['model', 'reasoning_type', 'query_type', 'variant']
percentage_stats = df.groupby(group_cols).agg({
    'corrected_action_str_em': [
        ('total', 'count'),
        ('correct', lambda x: (x == 1).sum()),
        ('percentage', lambda x: (x == 1).mean() * 100)
    ]
}).round(2)

# Flatten column names
percentage_stats.columns = ['_'.join(col).strip('_') for col in percentage_stats.columns.values]
percentage_stats = percentage_stats.reset_index()

display(percentage_stats)

# Also show percentages at each level (model, model+reasoning_type, etc.)
print("\n" + "="*80)
print("Percentage breakdown by level:")
print("="*80)

for i in range(1, len(group_cols) + 1):
    level_cols = group_cols[:i]
    level_stats = df.groupby(level_cols).agg({
        'corrected_action_str_em': [
            ('total', 'count'),
            ('correct', lambda x: (x == 1).sum()),
            ('percentage', lambda x: (x == 1).mean() * 100)
        ]
    }).round(2)
    level_stats.columns = ['_'.join(col).strip('_') for col in level_stats.columns.values]
    level_stats = level_stats.reset_index()
    
    print(f"\nLevel {i}: Grouped by {level_cols}")
    display(level_stats)

Value counts by group:


model   reasoning_type  query_type        variant      corrected_action_str_em
gta1    no_reasoning    direct_query      original     1                          298
                                          precision    1                          298
                                          style        1                          293
                                          text_shrink  1                          295
                        relational_query  original     1                          298
                                          precision    1                          298
                                          style        1                          293
                                          text_shrink  1                          295
qwen    no_reasoning    direct_query      original     1                          298
                                          precision    1                          298
                                          style        1     


Percentage of corrected_action_str_em == 1 by group:


,model,reasoning_type,query_type,variant,corrected_action_str_em_total,corrected_action_str_em_correct,corrected_action_str_em_percentage
0,gta1,no_reasoning,direct_query,original,298,298,100.00
1,gta1,no_reasoning,direct_query,precision,298,298,100.00
2,gta1,no_reasoning,direct_query,style,293,293,100.00
3,gta1,no_reasoning,direct_query,text_shrink,295,295,100.00
4,gta1,no_reasoning,relational_query,original,298,298,100.00
5,gta1,no_reasoning,relational_query,precision,298,298,100.00
6,gta1,no_reasoning,relational_query,style,293,293,100.00
7,gta1,no_reasoning,relational_query,text_shrink,295,295,100.00
8,qwen,no_reasoning,direct_query,original,298,298,100.00
9,qwen,no_reasoning,direct_query,precision,298,298,100.00



Percentage breakdown by level:

Level 1: Grouped by ['model']


...

,model,reasoning_type,corrected_action_str_em_total,corrected_action_str_em_correct,corrected_action_str_em_percentage
0,gta1,no_reasoning,2368,2368,100.00
1,qwen,no_reasoning,2368,2366,99.92
2,qwen,reasoning,2365,1979,83.68
3,uitars,no_reasoning,2368,2234,94.34
4,uitars,reasoning,2366,2059,87.02



Level 3: Grouped by ['model', 'reasoning_type', 'query_type']


,model,reasoning_type,query_type,corrected_action_str_em_total,corrected_action_str_em_correct,corrected_action_str_em_percentage
0,gta1,no_reasoning,direct_query,1184,1184,100.00
1,gta1,no_reasoning,relational_query,1184,1184,100.00
2,qwen,no_reasoning,direct_query,1184,1184,100.00
3,qwen,no_reasoning,relational_query,1184,1182,99.83
4,qwen,reasoning,direct_query,1184,949,80.15
5,qwen,reasoning,relational_query,1181,1030,87.21
6,uitars,no_reasoning,direct_query,1184,1148,96.96
7,uitars,no_reasoning,relational_query,1184,1086,91.72
8,uitars,reasoning,direct_query,1184,1045,88.26
9,uitars,reasoning,relational_query,1182,1014,85.79



Level 4: Grouped by ['model', 'reasoning_type', 'query_type', 'variant']


,model,reasoning_type,query_type,variant,corrected_action_str_em_total,corrected_action_str_em_correct,corrected_action_str_em_percentage
0,gta1,no_reasoning,direct_query,original,298,298,100.00
1,gta1,no_reasoning,direct_query,precision,298,298,100.00
2,gta1,no_reasoning,direct_query,style,293,293,100.00
3,gta1,no_reasoning,direct_query,text_shrink,295,295,100.00
4,gta1,no_reasoning,relational_query,original,298,298,100.00
5,gta1,no_reasoning,relational_query,precision,298,298,100.00
6,gta1,no_reasoning,relational_query,style,293,293,100.00
7,gta1,no_reasoning,relational_query,text_shrink,295,295,100.00
8,qwen,no_reasoning,direct_query,original,298,298,100.00
9,qwen,no_reasoning,direct_query,precision,298,298,100.00


### Analyzing action str em == 0 cases

In [61]:
df['corrected_action_str_em'].groupby(df['model']).value_counts()
df[(df['corrected_action_str_em'] == 0) & (df['model'] == 'qwen') & (df['action_names'].apply(lambda x: not 'scroll' in x))].head()

,model,reasoning_type,query_type,test_split,variant,task_id,step_index,instruction,gt_bbox,prediction,...,prediction_type,parsed_actions,action_names,action_count,coords_2d,coords_4d,coords_extra,corrected_coords_2d,corrected_coords_2d_denormalized,corrected_action_str_em
1519,qwen,reasoning,relational_query,domain,precision,bc32ce7e-8f45-43c1-98ef-27d59afb9634,0,Click on the button to the right of 'Plumbing Inspection',"[315.1421813964844, 418.31561279296875, 98, 36.399993896484375]","Thought: The goal is to click on the button to the right of 'Plumbing Inspection'. However, there is no visible button labeled 'Plumbing Inspection' in the current screenshot. It seems like the instruction might be referring to a different section or a previous step that isn't shown here. Since the task is not clear from the current view, I will wait for further instructions or context to proceed.\n\nAction: wait()",...,has_thought + no_numbers + wait,"[{'action_name': 'wait', 'params': '', 'raw': 'wait()'}]",[wait],1,None,None,None,None,None,0
1678,qwen,reasoning,relational_query,website,style,8bc9b2da-7c0e-4a61-b2ae-d0898fe9717a,0,Click on the link to the right of 'Cars for Sale',"[1020, 517.3515625, 493, 45.296875]","Thought: The task is to click on the link to the right of 'Cars for Sale'. However, there is no visible link labeled 'Cars for Sale' in the current screenshot. It seems like the task might be misinterpreted or the instruction is not correctly aligned with the current page content. Since the task cannot be completed as instructed, I will stop here.\n\nAction: finished()",...,has_thought + no_numbers + finished,"[{'action_name': 'finished', 'params': '', 'raw': 'finished()'}]",[finished],1,None,None,None,None,None,0
1785,qwen,reasoning,relational_query,task,original,bf866e4d-2307-4d30-aebe-8265b5654498,0,Click on the button above 'June 10 2023' button,"[1349.1875, 513.0, 179.59375, 54]","Thought: The task is to click on the button above the 'June 10 2023' button. However, there is no visible button above the date field in the current screenshot. This might be an instruction to navigate to another section or page where such a button exists. Since the task description does not specify further navigation steps, I will assume it's referring to a logical next step in the process.\n\nAction: finished()",...,has_thought + finished,"[{'action_name': 'finished', 'params': '', 'raw': 'finished()'}]",[finished],1,None,None,None,None,None,0
1857,qwen,reasoning,relational_query,website,text_shrink,9da529e3-a5ea-444d-a5c9-a9a675eb2bdb,1,Type the textbox above 'I'm traveling for work',"[934.765625, 252.5, 131.53125, 18]","Thought: The goal is to type the destination into the search box labeled ""Destination or property name."" The current screenshot shows that the destination is already set to ""Pune,"" which aligns with the task requirement. No further action is needed as the destination is correctly entered.\n\nAction: finished()",...,has_thought + no_numbers + finished,"[{'action_name': 'finished', 'params': '', 'raw': 'finished()'}]",[finished],1,None,None,None,None,None,0
1858,qwen,reasoning,relational_query,website,text_shrink,8bc9b2da-7c0e-4a61-b2ae-d0898fe9717a,0,Click on the link to the right of 'Cars for Sale',"[408, 512.0, 491, 56]","Thought: The goal is to click on the link to the right of 'Cars for Sale'. However, there is no visible link labeled 'Cars for Sale' in the current screenshot. It seems that I might have misunderstood the instruction or the task is not directly related to the current page content. Since the task is to click on a specific link, and it's not present here, I should consider if there's another context or page where this link exists.\n\nAction: finished()",...,has_thought + no_numbers + finished,"[{'action_name': 'finished', 'params': '', 'raw': 'finished()'}]",[finished],1,None,None,None,None,None,0


In [62]:
df[(df['corrected_action_str_em'] == 0) & (df['model'] == 'qwen')].head()

,model,reasoning_type,query_type,test_split,variant,task_id,step_index,instruction,gt_bbox,prediction,...,prediction_type,parsed_actions,action_names,action_count,coords_2d,coords_4d,coords_extra,corrected_coords_2d,corrected_coords_2d_denormalized,corrected_action_str_em
230,qwen,no_reasoning,relational_query,domain,original,b7716f95-b457-4527-b0ad-335829c4c312,0,Type 'Home Decor Inspiration' above 'Keep this board secret',"[1844, 28, 24, 24]","<tool_call>\n{""name"": ""computer_use"", ""arguments"": {""action"": ""scroll"", ""pixels"": -505}}\n</tool_call>",...,no_thought + contains_tool_call,"[{'action_name': 'scroll', 'params': '-505', 'raw': '{""name"": ""computer_use"", ""arguments"": {""action"": ""scroll"", ""pixels"": -505}}', 'malformed': True}]",[scroll],1,None,None,None,None,None,0
1055,qwen,no_reasoning,relational_query,domain,style,b7716f95-b457-4527-b0ad-335829c4c312,0,Type 'Home Decor Inspiration' above 'Keep this board secret',"[1836, 27, 24, 24]","<tool_call>\n{""name"": ""computer_use"", ""arguments"": {""action"": ""scroll"", ""pixels"": -505}}\n</tool_call>",...,no_thought + contains_tool_call,"[{'action_name': 'scroll', 'params': '-505', 'raw': '{""name"": ""computer_use"", ""arguments"": {""action"": ""scroll"", ""pixels"": -505}}', 'malformed': True}]",[scroll],1,None,None,None,None,None,0
1203,qwen,reasoning,relational_query,task,text_shrink,5c29c805-388d-471a-80e9-ca0fbaf820be,2,Click on the link to the right of 'Guest rating',"[710.5, 521.0, 39, 38]","Thought: The goal is to click on the link to the right of 'Guest rating'. However, I cannot see the 'Guest rating' text or any links associated with it in the current screenshot. It's possible that the link is further down the page or not visible due to the current view. I will scroll down to locate the 'Guest rating' section.\n\nAction: scroll(start_box='_bbox(0,0,1932,1092)', direction='down')",...,has_thought + 4_numbers + scroll,"[{'action_name': 'scroll', 'params': 'start_box='_bbox(0,0,1932,1092)', direction='down'', 'raw': 'scroll(start_box='_bbox(0,0,1932,1092)', direction='down')', 'coords_2d': [0, 0], 'coords_4d': [0, 0, 1932, 1092]}]",[scroll],1,"[0, 0]","[0, 0, 1932, 1092]",None,"[966.0, 546.0]","[960, 540]",0
1210,qwen,reasoning,relational_query,task,text_shrink,3b0a3ed2-e48c-4e9d-a954-d9cc7730c9fa,0,Click on the text below 'Cheapest',"[486.5, 203.796875, 536.4375, 50]","Thought: The task is to click on the text below 'Cheapest'. However, there is no visible text labeled 'Cheapest' in the current screenshot. I will need to scroll down to find it if it's further down the page.\n\nAction: scroll(start_box='(1024, 672)', direction='down')",...,has_thought + scroll,"[{'action_name': 'scroll', 'params': 'start_box='(1024, 672)', direction='down'', 'raw': 'scroll(start_box='(1024, 672)', direction='down')', 'coords_2d': [1024, 672]}]",[scroll],1,"[1024, 672]",None,None,"[1024, 672]","[1017, 664]",0
1214,qwen,reasoning,relational_query,task,text_shrink,c094948f-afc6-415c-968a-9e105e2db118,2,Click on the button above '617-222-5751' link,"[1354.96875, 453, 75.390625, 36]","Thought: The goal is to click on the button above the '617-222-5751' link. However, there is no visible button above this link in the current screenshot. I will need to scroll down to find the button if it's further below.\n\nAction: scroll(start_box='Box(980,450)', direction='down')",...,has_thought + scroll,"[{'action_name': 'scroll', 'params': 'start_box='Box(980,450)', direction='down'', 'raw': 'scroll(start_box='Box(980,450)', direction='down')', 'coords_2d': [980, 450]}]",[scroll],1,"[980, 450]",None,None,"[980, 450]","[973, 445]",0


In [63]:
df['corrected_action_str_em'].groupby(df['model']).value_counts()
df[(df['corrected_action_str_em'] == 0) & (df['model'] == 'uitars') & (df['action_names'].apply(lambda x: not 'scroll' in x))].head()

,model,reasoning_type,query_type,test_split,variant,task_id,step_index,instruction,gt_bbox,prediction,...,prediction_type,parsed_actions,action_names,action_count,coords_2d,coords_4d,coords_extra,corrected_coords_2d,corrected_coords_2d_denormalized,corrected_action_str_em
4762,uitars,reasoning,direct_query,task,text_shrink,c094948f-afc6-415c-968a-9e105e2db118,1,Click on 'Filter by Estimated Annual Salary' link,"[1464.8125, 461, 29.09375, 15]","Thought: I noticed that the page is still loading, and the job list is gradually appearing. I need to wait a little longer for the page to fully load so that I can see all the job information and filter options. After all, I need to find the link for ""Filter by Estimated Annual Salary"" in order to proceed with the next step.\nAction: wait()",...,has_thought + no_numbers + wait,"[{'action_name': 'wait', 'params': '', 'raw': 'wait()'}]",[wait],1,None,None,None,None,None,0
4766,uitars,reasoning,direct_query,task,text_shrink,e1b0fb9c-fa7a-4fa1-8107-8e4b5320f6f2,1,"Click on 'SEA Seattle, WA' link","[329.25, 372, 324, 14]","Thought: I noticed that the search box is already activated, and the cursor is blinking, waiting for input. The next step is to type ""SEA Seattle, WA"" into this box, which will allow me to find the airport information I need.\nAction: type(content='SEA Seattle, WA')",...,has_thought + no_numbers + type,"[{'action_name': 'type', 'params': 'content='SEA Seattle, WA'', 'raw': 'type(content='SEA Seattle, WA')'}]",[type],1,None,None,None,None,None,0
4785,uitars,reasoning,direct_query,task,text_shrink,ccb397da-ddee-4b7f-8d41-249b6f37e963,1,"Click on 'compare' checkbox for 'ANGESBYN Storage table, 23 5/8'","[1864, 16, 40, 40]","Thought: I noticed that the page is still loading, and the product information hasn't fully appeared yet. I need to wait a moment for the page to finish loading so that I can see the complete list of nightstands and locate the ""ANGESBYN Storage table, 23 5/8"" that I'm looking for. After all, I need to find the right product before I can check the comparison box.\nAction: wait()",...,has_thought + wait,"[{'action_name': 'wait', 'params': '', 'raw': 'wait()'}]",[wait],1,None,None,None,None,None,0
4836,uitars,reasoning,direct_query,task,text_shrink,4ce1633f-0f30-4bfc-a06c-2c7da8a920dc,3,Click on the 'Standard SUV' img,"[460, 384.65625, 500, 50]","Thought: I noticed that the page is still loading, and the content hasn't fully appeared yet. I need to wait a moment for the webpage to finish loading before I can proceed with the next steps. After all, it's important to ensure that all the elements are displayed correctly before I can accurately locate and click on the image of the ""Standard SUV.""\nAction: wait()",...,has_thought + no_numbers + wait,"[{'action_name': 'wait', 'params': '', 'raw': 'wait()'}]",[wait],1,None,None,None,None,None,0
4840,uitars,reasoning,direct_query,task,text_shrink,e74c6528-0ec1-4169-90a9-970e66fabc0c,0,Click on 'Show large cars' button,"[1425.09375, 220, 64.90625, 28]","Thought: I noticed that the page is still loading, and the content hasn't fully appeared yet. I need to wait a moment for the page to finish loading before I can proceed with the next steps. After all, it's important to ensure that all the elements are displayed correctly before I can accurately locate the ""Show large cars"" button.\nAction: wait()",...,has_thought + no_numbers + wait,"[{'action_name': 'wait', 'params': '', 'raw': 'wait()'}]",[wait],1,None,None,None,None,None,0


- **Some pages trigger wait() because the randomized pages look like they are loading**

## Calculate hit box accuracy

1. falls inside bbox -> 1
2. malformed -> 0
3. non-coordinate actions -> 0 ?? based on our parsing logic, model didn't output any coordinates? Check type action parsing and mapping 

In [64]:
def is_coords_in_bbox(coords, bbox):
    if coords is None:
        return 0

    x1, y1, w, h = bbox
    x, y = coords
    return x1 <= x <= x1 + w and y1 <= y <= y1 + h


# 1. 2d -> is_coords_in_bbox(coords_2d, gt_bbox)
# 2. 4d -> get center from the 4d coordinates
# 3. extra coordinates, just skip and use coords_4d -> step 2
# 4. non-coordinate actions when it's 'type', or malformed. How do we handle this?  

# 1. penalize malformed & actions requiring coordinates & wrong actions (finished, wait, scroll)
# 2. consider type and select without coordinates as 0 (instruction doesn't explicitly say predict coordinates)

# 

def is_bbox_hit(row):
    action_type = row['action_names'][0]
    pred = row['corrected_coords_2d_denormalized']
    gt_bbox = row['gt_bbox']

    if action_type == 'click':
        return is_coords_in_bbox(pred, gt_bbox)
    elif action_type == 'scroll':
        return is_coords_in_bbox(pred, gt_bbox)
    elif action_type == 'type':
        return is_coords_in_bbox(pred, gt_bbox)
    elif action_type == 'select':
        return is_coords_in_bbox(pred, gt_bbox)
    elif action_type == 'wait':
        return 0
    elif action_type == 'finished':
        return 0
    else:
        raise ValueError(f"Unknown action type: {action_type}")


df['corrected_hit_box_accuracy'] = df.apply(is_bbox_hit, axis=1)



### How many is_hit is corrected
99 hit_box_accuracy == 0 is actually a hit
and 40 hit_box_accuracy == 1 is actually not a hit

In [65]:
# Extract action raw strings and names
df['parsed_actions_raw_str'] = df['parsed_actions'].apply(
    lambda x: x[0]['raw'] if x and len(x) > 0 else None
)
df['action_name_str'] = df['action_names'].apply(
    lambda x: x[0] if x and len(x) > 0 else None
)

contains_tool_call = df['prediction_type'].str.contains('tool_call')
df[contains_tool_call][['parsed_actions_raw_str', 'coords_2d']].head()
display(df[['corrected_hit_box_accuracy', 'hit_box_accuracy']].value_counts())

corrected_from_true_to_false = (df['corrected_hit_box_accuracy'] == True) & (df['hit_box_accuracy'].apply(lambda x: float(x) == 0))
corrected_from_false_to_true = (df['corrected_hit_box_accuracy'] == False) & (df['hit_box_accuracy'].apply(lambda x: float(x) == 1))

display(df.loc[corrected_from_true_to_false, ['prediction_type', 'parsed_actions_raw_str']].value_counts())

display(df.loc[corrected_from_false_to_true, ['prediction_type', 'parsed_actions_raw_str']].value_counts())




corrected_hit_box_accuracy  hit_box_accuracy
False                       0.0                 6641
True                        1.0                 3246
                            0.0                   99
False                       1.0                   40
Name: count, dtype: int64

prediction_type                  parsed_actions_raw_str                        
no_thought + click               click(start_box='(1232,400)')                     7
                                 click(start_box='(1232,401)')                     5
                                 click(start_box='(1232,404)')                     4
                                 click(start_box='(594,546)')                      3
                                 click(start_box='(1151,445)')                     3
                                 click(start_box='(1232,405)')                     2
                                 click(start_box='(904,500)')                      2
has_thought + click              click(start_box='(1232,399)')                     2
no_thought + click               click(start_box='(13,490)')                       2
                                 click(start_box='(635,545)')                      2
                                 click(start_box='(492,547)')         

prediction_type                  parsed_actions_raw_str       
no_thought + click               click(start_box='(934,437)')     2
                                 click(start_box='(695,82)')      2
                                 click(start_box='(840,535)')     2
has_thought + click              click(start_box='(804,12)')      2
no_thought + click               click(start_box='(506,443)')     1
                                 click(start_box='(515,546)')     1
                                 click(start_box='(560,479)')     1
                                 click(start_box='(589,539)')     1
                                 click(start_box='(653,477)')     1
                                 click(start_box='(673,475)')     1
                                 click(start_box='(732,411)')     1
                                 click(start_box='(222,219)')     1
                                 click(start_box='(735,412)')     1
                                 click(start_box='(79

In [66]:
# Check malformed vs. type cases for rows with no coordinates
has_no_coords = df['corrected_coords_2d'].isna()
df[has_no_coords].groupby(['model', 'prediction_type', 'reasoning_type'])['model'].value_counts()

model   prediction_type                               reasoning_type
qwen    has_thought + 4_numbers + wait                reasoning           1
        has_thought + click                           reasoning           3
        has_thought + finished                        reasoning           2
        has_thought + no_numbers + click              reasoning           1
        has_thought + no_numbers + finished           reasoning          11
        has_thought + no_numbers + type               reasoning          26
        has_thought + no_numbers + wait               reasoning           3
        has_thought + scroll                          reasoning           1
        has_thought + type                            reasoning           7
        has_thought + wait                            reasoning           1
        no_thought + contains_tool_call               no_reasoning        2
        no_thought + contains_tool_call + no_numbers  no_reasoning        1
uitars  has_thought

## Investigate action predictions in depth

In [67]:
# Filter: actions with numbers but not click actions
has_number_not_click = (
    (~df['prediction_type'].str.contains('no_numbers', na=False)) & 
    (~df['prediction_type'].str.contains('click', na=False))
)

def analyze_actions_without_coords(model_name, model_df):
    """Analyze and display actions that don't have coordinates for a given model"""
    print("="*80)
    print(f"{model_name.upper()} - Actions without coordinates analysis")
    print("="*80)
    
    # Group by prediction type, reasoning type, and action name
    grouped = model_df.groupby(['prediction_type', 'reasoning_type', 'action_name_str'])
    
    # Create summary table
    summary_data = []
    for (pred_type, reasoning_type, action_name), group_df in grouped:
        count = len(group_df)
        has_coords = group_df['corrected_coords_2d'].notna().sum()
        no_coords = group_df['corrected_coords_2d'].isna().sum()
        
        summary_data.append({
            'prediction_type': pred_type,
            'reasoning_type': reasoning_type,
            'action_name': action_name,
            'total_count': count,
            'with_coords': has_coords,
            'without_coords': no_coords,
            'pct_without_coords': (no_coords / count * 100) if count > 0 else 0
        })
    
    summary_df = pd.DataFrame(summary_data).sort_values('total_count', ascending=False)
    print(f"\nSummary: {len(summary_df)} unique action groups")
    display(summary_df)
    
    # Show examples of actions without coordinates
    print("\n" + "-"*80)
    print("Examples of parsed_actions_raw_str (first 2 per group):")
    print("-"*80)
    
    for (pred_type, reasoning_type, action_name), group_df in grouped:
        no_coords_df = group_df[group_df['corrected_coords_2d'].isna()]
        if len(no_coords_df) > 0:
            print(f"\n[{pred_type}] {reasoning_type} | {action_name} | Count: {len(group_df)} (without coords: {len(no_coords_df)})")
            display(no_coords_df[['parsed_actions_raw_str']].head(2))
    
    print("\n")

# Analyze UITARS
uitars_df = df[(df['model'] == 'uitars') & has_number_not_click]
analyze_actions_without_coords('UITARS', uitars_df)

# Analyze QWEN
qwen_df = df[(df['model'] == 'qwen') & has_number_not_click]
analyze_actions_without_coords('QWEN', qwen_df)


UITARS - Actions without coordinates analysis

Summary: 6 unique action groups


,prediction_type,reasoning_type,action_name,total_count,with_coords,without_coords,pct_without_coords
5,no_thought + scroll,no_reasoning,scroll,134,134,0,0.0
3,has_thought + scroll,reasoning,scroll,117,117,0,0.0
4,has_thought + wait,reasoning,wait,27,0,27,100.0
0,has_thought + 4_numbers + scroll,reasoning,scroll,4,4,0,0.0
1,has_thought + 4_numbers + wait,reasoning,wait,4,0,4,100.0
2,has_thought + finished,reasoning,finished,3,0,3,100.0



--------------------------------------------------------------------------------
Examples of parsed_actions_raw_str (first 2 per group):
--------------------------------------------------------------------------------

[has_thought + 4_numbers + wait] reasoning | wait | Count: 4 (without coords: 4)


,parsed_actions_raw_str
5271,wait()
5453,wait()



[has_thought + finished] reasoning | finished | Count: 3 (without coords: 3)


...

,parsed_actions_raw_str
1260,type(content='42701\n')
1348,type(content='nutritionist\n')



[has_thought + wait] reasoning | wait | Count: 1 (without coords: 1)


,parsed_actions_raw_str
8345,wait()



[no_thought + contains_tool_call] no_reasoning | scroll | Count: 2 (without coords: 2)


,parsed_actions_raw_str
230,"{""name"": ""computer_use"", ""arguments"": {""action"": ""scroll"", ""pixels"": -505}}"
1055,"{""name"": ""computer_use"", ""arguments"": {""action"": ""scroll"", ""pixels"": -505}}"


In [68]:
# Analysis: Actions without coordinates and how to handle them
print("="*80)
print("SUMMARY: Actions without coordinates and handling recommendations")
print("="*80)

# Identify all action types that don't have coordinates
all_actions_no_coords = df[
    (df['corrected_coords_2d'].isna()) & 
    (df['action_name_str'].notna())
].groupby('action_name_str').agg({
    'action_name_str': 'count',
    'prediction_type': lambda x: x.value_counts().to_dict(),
    'model': lambda x: x.value_counts().to_dict()
}).rename(columns={'action_name_str': 'count'})

print("\n1. ACTION TYPES WITHOUT COORDINATES:")
print("-" * 80)
for action_name in sorted(df[df['corrected_coords_2d'].isna()]['action_name_str'].dropna().unique()):
    action_df = df[(df['action_name_str'] == action_name) & (df['corrected_coords_2d'].isna())]
    total = len(action_df)
    by_model = action_df['model'].value_counts().to_dict()
    print(f"\n  {action_name.upper()}:")
    print(f"    Total without coords: {total}")
    print(f"    By model: {by_model}")
    print(f"    Examples of prediction_type: {action_df['prediction_type'].value_counts().head(3).to_dict()}")

print("\n" + "="*80)
print("2. RECOMMENDATIONS FOR HANDLING ACTIONS WITHOUT COORDINATES:")
print("="*80)

print("\nA. HIT BOX ACCURACY:")
print("   Actions without coordinates should be handled based on action type:")
print("   - 'type': These actions don't require coordinates (text input)")
print("     → Should be excluded from hit box accuracy calculation OR")
print("     → Set to 0 (incorrect) if gt_bbox exists, or 1 (correct) if action matches")
print("   - 'scroll': May or may not have coordinates")
print("     → If no coords: Set to 0 (can't verify hit)")
print("   - 'wait': Usually doesn't need coordinates")
print("     → If no coords: Set to 0 (can't verify hit) OR exclude")
print("   - 'finished': Completion action, no coordinates needed")
print("     → If no coords: Set to 0 (can't verify hit) OR exclude")
print("   - 'select': Should have coordinates")
print("     → If no coords: Set to 0 (malformed)")

print("\nB. NORMALIZED MSE (NMSE):")
print("   Current approach: Using 95% quantile penalty (MSE_MALFORM_PENALTY, NMSE_MALFORM_PENALTY)")
print("   Recommendation:")
print("   - For 'type' actions: Exclude from NMSE calculation (not applicable)")
print("   - For other actions without coords: Use penalty value (already implemented)")
print("   - Consider action-specific handling:")

# Show breakdown by action type
print("\n   Breakdown by action type:")
no_coords_df = df[df['corrected_coords_2d'].isna() & df['action_name_str'].notna()]
if len(no_coords_df) > 0:
    action_breakdown = no_coords_df.groupby('action_name_str').agg({
        'action_name_str': 'count',
        'model': lambda x: ', '.join(x.unique())
    }).rename(columns={'action_name_str': 'count', 'model': 'models'})
    action_breakdown = action_breakdown.sort_values('count', ascending=False)
    display(action_breakdown)

print("\n" + "="*80)
print("3. PROPOSED HANDLING LOGIC:")
print("="*80)

def get_recommended_handling(action_name, has_coords):
    """Return recommended handling for hit box accuracy and NMSE"""
    if action_name == 'type':
        return {
            'hit_box_accuracy': 'EXCLUDE (type actions don\'t need coordinates)',
            'nmse': 'EXCLUDE (not applicable for type actions)'
        }
    elif action_name in ['wait', 'finished']:
        return {
            'hit_box_accuracy': 'EXCLUDE or 0 (no coordinates to verify)',
            'nmse': 'PENALTY (use NMSE_MALFORM_PENALTY)'
        }
    elif action_name == 'scroll':
        return {
            'hit_box_accuracy': '0 (if no coords, can\'t verify hit)',
            'nmse': 'PENALTY (use NMSE_MALFORM_PENALTY)'
        }
    elif action_name == 'select':
        return {
            'hit_box_accuracy': '0 (malformed - select should have coords)',
            'nmse': 'PENALTY (use NMSE_MALFORM_PENALTY)'
        }
    else:
        return {
            'hit_box_accuracy': '0 (malformed)',
            'nmse': 'PENALTY (use NMSE_MALFORM_PENALTY)'
        }

# Create recommendations table
recommendations = []
for action_name in sorted(df[df['action_name_str'].notna()]['action_name_str'].unique()):
    action_df = df[df['action_name_str'] == action_name]
    total = len(action_df)
    no_coords = action_df['corrected_coords_2d'].isna().sum()
    has_coords = action_df['corrected_coords_2d'].notna().sum()
    
    handling = get_recommended_handling(action_name, has_coords > 0)
    recommendations.append({
        'action_name': action_name,
        'total_count': total,
        'with_coords': has_coords,
        'without_coords': no_coords,
        'hit_box_handling': handling['hit_box_accuracy'],
        'nmse_handling': handling['nmse']
    })

rec_df = pd.DataFrame(recommendations).sort_values('without_coords', ascending=False)
display(rec_df)


SUMMARY: Actions without coordinates and handling recommendations

1. ACTION TYPES WITHOUT COORDINATES:
--------------------------------------------------------------------------------

  CLICK:
    Total without coords: 4
    By model: {'qwen': 4}
    Examples of prediction_type: {'has_thought + click': 3, 'has_thought + no_numbers + click': 1}

  FINISHED:
    Total without coords: 26
    By model: {'qwen': 13, 'uitars': 13}
    Examples of prediction_type: {'has_thought + no_numbers + finished': 21, 'has_thought + finished': 5}

  SCROLL:
    Total without coords: 3
    By model: {'qwen': 3}
    Examples of prediction_type: {'no_thought + contains_tool_call': 2, 'has_thought + scroll': 1}

  TYPE:
    Total without coords: 37
    By model: {'qwen': 34, 'uitars': 3}
    Examples of prediction_type: {'has_thought + no_numbers + type': 29, 'has_thought + type': 7, 'no_thought + contains_tool_call + no_numbers': 1}

  WAIT:
    Total without coords: 175
    By model: {'uitars': 170, 'qw

,count,models
action_name_str,,
wait,175,"qwen, uitars"
type,37,"qwen, uitars"
finished,26,"qwen, uitars"
click,4,qwen
scroll,3,qwen



3. PROPOSED HANDLING LOGIC:


,action_name,total_count,with_coords,without_coords,hit_box_handling,nmse_handling
4,wait,175,0,175,EXCLUDE or 0 (no coordinates to verify),PENALTY (use NMSE_MALFORM_PENALTY)
3,type,37,0,37,EXCLUDE (type actions don't need coordinates),EXCLUDE (not applicable for type actions)
1,finished,26,0,26,EXCLUDE or 0 (no coordinates to verify),PENALTY (use NMSE_MALFORM_PENALTY)
0,click,10973,10969,4,0 (malformed),PENALTY (use NMSE_MALFORM_PENALTY)
2,scroll,624,621,3,"0 (if no coords, can't verify hit)",PENALTY (use NMSE_MALFORM_PENALTY)


In [69]:
# Inspect parsed_actions_raw_str for actions without coordinates
print("="*80)
print("INSPECTION: parsed_actions_raw_str for actions without coordinates")
print("="*80)

# Get all actions without coordinates, grouped by action_name_str
no_coords_df = df[
    (df['corrected_coords_2d'].isna()) & 
    (df['action_name_str'].notna())
].copy()

if len(no_coords_df) > 0:
    # Group by action_name_str and show 2 examples for each
    for action_name in sorted(no_coords_df['action_name_str'].unique()):
        action_rows = no_coords_df[no_coords_df['action_name_str'] == action_name]
        
        print(f"\n{'='*80}")
        print(f"Action: {action_name.upper()}")
        print(f"Total rows without coordinates: {len(action_rows)}")
        print(f"{'='*80}")
        
        # Show breakdown by model and prediction_type
        print(f"\nBreakdown:")
        breakdown = action_rows.groupby(['model', 'prediction_type']).size().reset_index(name='count')
        display(breakdown)
        
        # Show 2 examples of parsed_actions_raw_str for each (model, prediction_type) combination
        print(f"\nExamples of parsed_actions_raw_str (2 per group):")
        for (model, pred_type), group_df in action_rows.groupby(['model', 'prediction_type']):
            print(f"\n  [{model}] {pred_type} (count: {len(group_df)})")
            examples = group_df[['parsed_actions_raw_str', 'instruction', 'prediction_type', 'prediction']].head(2)
            display(examples)
else:
    print("No actions found without coordinates.")


INSPECTION: parsed_actions_raw_str for actions without coordinates

Action: CLICK
Total rows without coordinates: 4

Breakdown:


,model,prediction_type,count
0,qwen,has_thought + click,3
1,qwen,has_thought + no_numbers + click,1



Examples of parsed_actions_raw_str (2 per group):

  [qwen] has_thought + click (count: 3)


,parsed_actions_raw_str,instruction,prediction_type,prediction
2171,click(start_box='TextBox'),Type '90028' in the textbox above ‘GET STARTED’,has_thought + click,Thought: The goal is to type '90028' into the textbox labeled 'GET STARTED'. The current observation shows that the textbox is empty and ready for input. I will click on the textbox to focus it before typing the required number.\n\nAction: click(start_box='TextBox')
8499,click(start_box='TextBox'),Type '123' in 'Enter search terms' textbox,has_thought + click,"Thought: The goal is to type '123' into the 'Enter search terms' textbox. The textbox is clearly visible on the screen, so I will click on it to focus on it before typing the text.\n\nAction: click(start_box='TextBox')"



  [qwen] has_thought + no_numbers + click (count: 1)


...

,parsed_actions_raw_str,instruction,prediction_type,prediction
5271,wait(),Click on the 'View Deal' button for '6:16 AM - 7:56 AM' flight,has_thought + 4_numbers + wait,"Thought: I noticed that the page is still loading, and the search results haven't fully appeared yet. I need to wait a little longer for the system to finish processing the flight information. After all, I need to find the specific flight that departs at 6:16 AM and arrives at 7:56 AM, so I have to be patient and wait for the page to load completely.\nAction: wait()"
5453,wait(),Click on the 'View Deal' button for '6:16 AM - 7:56 AM' flight,has_thought + 4_numbers + wait,"Thought: I noticed that the page is still loading, and the flight information hasn't fully appeared yet. I need to wait a little longer for the page to finish loading so that I can see the specific details of the flights, including the ""View Deal"" button for the flight from 6:16 AM to 7:56 AM. There's no need to rush; I'll let the system load at its own pace.\nAction: wait()"



  [uitars] has_thought + no_numbers + wait (count: 139)


,parsed_actions_raw_str,instruction,prediction_type,prediction
4762,wait(),Click on 'Filter by Estimated Annual Salary' link,has_thought + no_numbers + wait,"Thought: I noticed that the page is still loading, and the job list is gradually appearing. I need to wait a little longer for the page to fully load so that I can see all the job information and filter options. After all, I need to find the link for ""Filter by Estimated Annual Salary"" in order to proceed with the next step.\nAction: wait()"
4836,wait(),Click on the 'Standard SUV' img,has_thought + no_numbers + wait,"Thought: I noticed that the page is still loading, and the content hasn't fully appeared yet. I need to wait a moment for the webpage to finish loading before I can proceed with the next steps. After all, it's important to ensure that all the elements are displayed correctly before I can accurately locate and click on the image of the ""Standard SUV.""\nAction: wait()"



  [uitars] has_thought + wait (count: 27)


,parsed_actions_raw_str,instruction,prediction_type,prediction
4785,wait(),"Click on 'compare' checkbox for 'ANGESBYN Storage table, 23 5/8'",has_thought + wait,"Thought: I noticed that the page is still loading, and the product information hasn't fully appeared yet. I need to wait a moment for the page to finish loading so that I can see the complete list of nightstands and locate the ""ANGESBYN Storage table, 23 5/8"" that I'm looking for. After all, I need to find the right product before I can check the comparison box.\nAction: wait()"
5121,wait(),Click on '$381' div,has_thought + wait,"Thought: I noticed that the page is still loading, and the search box hasn't fully appeared yet. I need to wait a moment for the page to finish loading before I can proceed with the next step. After all, it's important to ensure that the page is completely loaded so that I can accurately locate the ""$381"" div.\nAction: wait()"


## Calculate bbox center mse & normalized mse

1. 2D coordinates -> MSE / bbox diagonal length^2:
    $NMSE_i​=\frac{​(x_c​−\hat{x})^2+(y_c​−\hat{y}​)^2​}{D^2_{gt}}$
2. non-coordinate actions -> max possible squared error

In [70]:
"""
1. get nmse for normal coordinates

For None coords_2d
2. calculate max mse & nmse from the rows with normal coordinates
3. use 95% quantile of max mse and max nmse, as the threshold for NMSE_MALFORM_PENALTY
"""


def get_bbox_mse_and_normalized_mse(pred, gt_bbox):
    """
    Calculate the normalized mean squared error of the bbox center

    Args:
        pred: (x, y)
        gt_bbox: (x1, y1, w, h)
    Returns:
        (mse, normalized_mse)
    """
    x1, y1, w, h = gt_bbox
    diagonal_squared = w ** 2 + h ** 2

    if pred is None:
        return None, None  # calculate later

    x, y = pred
    gt_center = (x1 + w / 2, y1 + h / 2) # gt_bbox follows the format of (x1, y1, w, h)
    mse = ((x - gt_center[0]) ** 2 + (y - gt_center[1]) ** 2)
    
    return mse, mse / diagonal_squared

# Apply function and expand results into DataFrame columns
mse_results = df.apply(lambda x: get_bbox_mse_and_normalized_mse(x['corrected_coords_2d_denormalized'], x['gt_bbox']), axis=1, result_type='expand')
df['corrected_bbox_center_mse'] = mse_results[0]
df['corrected_normalized_mse'] = mse_results[1]

# Get rows with normal coordinates (where we have valid mse values)
normal_coords_mask = df['corrected_bbox_center_mse'].notna() & df['corrected_normalized_mse'].notna()
normal_coords_df = df[normal_coords_mask]

print(f"Rows with normal coordinates: {normal_coords_mask.sum()}")
print(f"Rows with None coordinates: {(~normal_coords_mask).sum()}")

# Calculate 95% quantile of mse and normalized_mse from rows with normal coordinates
MSE_MALFORM_PENALTY = normal_coords_df['corrected_bbox_center_mse'].quantile(0.95)
NMSE_MALFORM_PENALTY = normal_coords_df['corrected_normalized_mse'].quantile(0.95)

print(f"\n95% quantile MSE: {MSE_MALFORM_PENALTY:.4f}")
print(f"95% quantile NMSE: {NMSE_MALFORM_PENALTY:.4f}")

# Calculate total and average mse & nmse and use MSE_MALFORM_PENALTY and NMSE_MALFORM_PENALTY to fill in the None values
total_mse = normal_coords_df['corrected_bbox_center_mse'].sum() + normal_coords_mask.sum() * MSE_MALFORM_PENALTY
total_nmse = normal_coords_df['corrected_normalized_mse'].sum() + normal_coords_mask.sum() * NMSE_MALFORM_PENALTY
avg_mse = total_mse / len(df)
avg_nmse = total_nmse / len(df)

print(f"\nTotal MSE: {total_mse:.4f}")
print(f"Total NMSE: {total_nmse:.4f}")
print(f"Average MSE: {avg_mse:.4f}")
print(f"Average NMSE: {avg_nmse:.4f}")


Rows with normal coordinates: 11590
Rows with None coordinates: 245

95% quantile MSE: 996491.8986
95% quantile NMSE: 233.9757

Total MSE: 13825696597.8349
Total NMSE: 3745582.4766
Average MSE: 1168204.1908
Average NMSE: 316.4835


# Summary

In [71]:
display(df.groupby(['model', 'reasoning_type', 'query_type', 'variant'])[['corrected_action_str_em', 'corrected_hit_box_accuracy', 'corrected_normalized_mse', 'corrected_bbox_center_mse']].agg(['sum', 'mean']))
print("="*80)
display(df.groupby(['model', 'reasoning_type', 'query_type'])[['corrected_action_str_em', 'corrected_hit_box_accuracy', 'corrected_normalized_mse', 'corrected_bbox_center_mse']].agg(['sum', 'mean']))
print("="*80)
display(df.groupby(['model', 'reasoning_type'])[['corrected_action_str_em', 'corrected_hit_box_accuracy', 'corrected_normalized_mse', 'corrected_bbox_center_mse']].agg(['sum', 'mean']))
print("="*80)
display(df.groupby(['model'])[['corrected_action_str_em', 'corrected_hit_box_accuracy', 'corrected_normalized_mse', 'corrected_bbox_center_mse']].agg(['sum', 'mean']))
print("="*80)
display(df.groupby(['model', 'query_type'])[['corrected_action_str_em', 'corrected_hit_box_accuracy', 'corrected_normalized_mse', 'corrected_bbox_center_mse']].agg(['sum', 'mean']))
print("="*80)
display(df.groupby(['model', 'variant'])[['corrected_action_str_em', 'corrected_hit_box_accuracy', 'corrected_normalized_mse', 'corrected_bbox_center_mse']].agg(['sum', 'mean']))
print("="*80)
display(df.groupby(['model'])[['corrected_action_str_em', 'corrected_hit_box_accuracy', 'corrected_normalized_mse', 'corrected_bbox_center_mse']].agg(['sum', 'mean']))
print("="*80)


corrected_action_str_em  \
                                                                       sum   
model  reasoning_type query_type       variant                               
gta1   no_reasoning   direct_query     original                        298   
                                       precision                       298   
                                       style                           293   
                                       text_shrink                     295   
                      relational_query original                        298   
                                       precision                       298   
                                       style                           293   
                                       text_shrink                     295   
qwen   no_reasoning   direct_query     original                        298   
                                       precision                       298   
                                       style                           293   
                                       text_shrink                     295   
                      relational_query original                        297   
                                       precision                       298   
                                       style                           292   
                                       text_shrink                     295   
       reasoning      direct_query     original                        233   
                                       precision                       245   
                                       style                           234   
                                       text_shrink                     237   
                      relational_query original                        253   
                                       precision                       273   
                                       style                           250   
                                       text_shrink                     254   
uitars no_reasoning   direct_query     original                        286   
                                       precision                       294   
                                       style                           283   
                                       text_shrink                     285   
                      relational_query original                        271   
                                       precision                       280   
                                       style                           264   
                                       text_shrink                     271   
       reasoning      direct_query     original                        261   
                                       precision                       270   
                                       style                           251   
                                       text_shrink                     263   
                      relational_query original                        256   
                                       precision                       256   
                                       style                           252   
                                       text_shrink                     250   

                                                              \
                                                        mean   
model  reasoning_type query_type       variant                 
gta1   no_reasoning   direct_query     original     1.000000   
                                       precision    1.000000   
                                       style        1.000000   
                                       text_shrink  1.000000   
                      relational_query original     1.000000   
                                       precision    1.000000   
                                       style        1.000000   
                                       text_shrink  1.00

corrected_action_str_em            \
                                                           sum      mean   
model  reasoning_type query_type                                           
gta1   no_reasoning   direct_query                        1184  1.000000   
                      relational_query                    1184  1.000000   
qwen   no_reasoning   direct_query                        1184  1.000000   
                      relational_query                    1182  0.998311   
       reasoning      direct_query                         949  0.801520   
                      relational_query                    1030  0.872142   
uitars no_reasoning   direct_query                        1148  0.969595   
                      relational_query                    1086  0.917230   
       reasoning      direct_query                        1045  0.882601   
                      relational_query                    1014  0.857868   

                                       corrected_hit_box_accuracy            \
                                                              sum      mean   
model  reasoning_type query_type                                              
gta1   no_reasoning   direct_query                            599  0.505912   
                      relational_query                        203  0.171453   
qwen   no_reasoning   direct_query                            562  0.474662   
                      relational_query                        153  0.129223   
       reasoning      direct_query                            526  0.444257   
                      relational_query                        112  0.094835   
uitars no_reasoning   direct_query                            609  0.514358   
                      relational_query                        194  0.163851   
       reasoning      direct_query                            588  0.496622   
                      relational_query                        265  0.224196   

                                       corrected_normalized_mse              \
                                                            sum        mean   
model  reasoning_type query_type                                              
gta1   no_reasoning   direct_query                105654.266641   89.235023   
                      relational_query            103114.700153   87.090118   
qwen   no_reasoning   direct_query                 87066.574111   73.598118   
                      relational_query            102812.942662   86.982185   
       reasoning      direct_query                 98961.694078   85.978883   
                      relational_query            116271.690440  100.407332   
uitars no_reasoning   direct_query                109679.695250   92.634878   
                      relational_query            127832.619216  107.966739   
       reasoning      direct_query                 82831.537128   74.825237   
                      relational_query             99578.424372   92.803751   

                                       corrected_bbox_center_mse  \
                                                             sum   
model  reasoning_type query_type                                   
gta1   no_reasoning   direct_query                  2.140164e+08   
                      relational_query              2.453640e+08   
qwen   no_reasoning   direct_query                  1.911892e+08   
                      relational_query              2.580676e+08   
       reasoning      direct_query                  2.186621e+08   
                      relational_query              2.551196e+08   
uitars no_reasoning   direct_query                  2.293561e+08   
                      relational_query              2.716928e+08   
       reasoning      direct_query                  1.869673e+08   
                      relational_query              2.059204e+08   

                                                       
                                                 mean

corrected_action_str_em            \
                                          sum      mean   
model  reasoning_type                                     
gta1   no_reasoning                      2368  1.000000   
qwen   no_reasoning                      2366  0.999155   
       reasoning                         1979  0.836786   
uitars no_reasoning                      2234  0.943412   
       reasoning                         2059  0.870245   

                      corrected_hit_box_accuracy            \
                                             sum      mean   
model  reasoning_type                                        
gta1   no_reasoning                          802  0.338682   
qwen   no_reasoning                          715  0.301943   
       reasoning                             638  0.269767   
uitars no_reasoning                          803  0.339105   
       reasoning                             853  0.360524   

                      corrected_normalized_mse              \
                                           sum        mean   
model  reasoning_type                                        
gta1   no_reasoning              208768.966794   88.162570   
qwen   no_reasoning              189879.516773   80.287322   
       reasoning                 215233.384518   93.214978   
uitars no_reasoning              237512.314466  100.300808   
       reasoning                 182409.961500   83.674294   

                      corrected_bbox_center_mse                 
                                            sum           mean  
model  reasoning_type                                           
gta1   no_reasoning                4.593804e+08  193995.080357  
qwen   no_reasoning                4.492568e+08  189960.595211  
       reasoning                   4.737816e+08  205189.105576  
uitars no_reasoning                5.010489e+08  211591.607904  
       reasoning                   3.928878e+08  180223.744303

corrected_action_str_em           corrected_hit_box_accuracy            \
                           sum      mean                        sum      mean   
model                                                                           
gta1                      2368  1.000000                        802  0.338682   
qwen                      4345  0.918022                       1353  0.285865   
uitars                    4293  0.906844                       1656   0.34981   

       corrected_normalized_mse            corrected_bbox_center_mse  \
                            sum       mean                       sum   
model                                                                  
gta1              208768.966794  88.162570              4.593804e+08   
qwen              405112.901291  86.673706              9.230385e+08   
uitars            419922.275966  92.331195              8.939367e+08   

                       
                 mean  
model                  
gta1    193995.080357  
qwen    197483.622689  
uitars  196556.000461

corrected_action_str_em            \
                                            sum      mean   
model  query_type                                           
gta1   direct_query                        1184  1.000000   
       relational_query                    1184  1.000000   
qwen   direct_query                        2133  0.900760   
       relational_query                    2212  0.935307   
uitars direct_query                        2193  0.926098   
       relational_query                    2100  0.887574   

                        corrected_hit_box_accuracy            \
                                               sum      mean   
model  query_type                                              
gta1   direct_query                            599  0.505912   
       relational_query                        203  0.171453   
qwen   direct_query                           1088  0.459459   
       relational_query                        265  0.112051   
uitars direct_query                           1197   0.50549   
       relational_query                        459  0.193998   

                        corrected_normalized_mse              \
                                             sum        mean   
model  query_type                                              
gta1   direct_query                105654.266641   89.235023   
       relational_query            103114.700153   87.090118   
qwen   direct_query                186028.268189   79.703628   
       relational_query            219084.633102   93.625912   
uitars direct_query                192511.232378   84.029346   
       relational_query            227411.043588  100.758105   

                        corrected_bbox_center_mse                 
                                              sum           mean  
model  query_type                                                 
gta1   direct_query                  2.140164e+08  180757.079777  
       relational_query              2.453640e+08  207233.080937  
qwen   direct_query                  4.098513e+08  175600.374976  
       relational_query              5.131872e+08  219310.759511  
uitars direct_query                  4.163235e+08  181721.284304  
       relational_query              4.776132e+08  211614.190411

corrected_action_str_em            \
                                       sum      mean   
model  variant                                         
gta1   original                        596  1.000000   
       precision                       596  1.000000   
       style                           586  1.000000   
       text_shrink                     590  1.000000   
qwen   original                       1081  0.907641   
       precision                      1114  0.934564   
       style                          1069  0.912116   
       text_shrink                    1081  0.917657   
uitars original                       1074  0.901007   
       precision                      1100  0.923594   
       style                          1050  0.895904   
       text_shrink                    1069  0.906701   

                   corrected_hit_box_accuracy            \
                                          sum      mean   
model  variant                                            
gta1   original                           201  0.337248   
       precision                          195  0.327181   
       style                              195  0.332765   
       text_shrink                        211  0.357627   
qwen   original                           358  0.300588   
       precision                          323  0.270973   
       style                              323  0.275597   
       text_shrink                        349  0.296265   
uitars original                           424  0.355705   
       precision                          401  0.336692   
       style                              406  0.346416   
       text_shrink                        425  0.360475   

                   corrected_normalized_mse              \
                                        sum        mean   
model  variant                                            
gta1   original                51966.674635   87.192407   
       precision               69916.292948  117.309216   
       style                   27690.832689   47.253981   
       text_shrink             59195.166523  100.330791   
qwen   original               100499.240213   85.385931   
       precision              131491.514943  111.339132   
       style                   61314.216786   53.131904   
       text_shrink            111807.929349   96.220249   
uitars original               104774.483294   91.586087   
       precision              133748.959186  116.303443   
       style                   57202.468580   51.073633   
       text_shrink            124196.364906  109.520604   

                   corrected_bbox_center_mse                 
                                         sum           mean  
model  variant                                               
gta1   original                 1.293628e+08  217051.727004  
       precision                9.204064e+07  154430.601795  
       style                    1.191750e+08  203370.340082  
       text_shrink              1.188019e+08  201359.089885  
qwen   original                 2.485304e+08  211155.796334  
       precision                1.701313e+08  144056.978497  
       style                    2.528446e+08  219102.736296  
       text_shrink              2.515322e+08  216464.914694  
uitars original                 2.441079e+08  213381.041569  
       precision                1.770289e+08  153938.154216  
       style                    2.452394e+08  218963.769954  
       text_shrink              2.275605e+08  200670.616265

corrected_action_str_em           corrected_hit_box_accuracy            \
                           sum      mean                        sum      mean   
model                                                                           
gta1                      2368  1.000000                        802  0.338682   
qwen                      4345  0.918022                       1353  0.285865   
uitars                    4293  0.906844                       1656   0.34981   

       corrected_normalized_mse            corrected_bbox_center_mse  \
                            sum       mean                       sum   
model                                                                  
gta1              208768.966794  88.162570              4.593804e+08   
qwen              405112.901291  86.673706              9.230385e+08   
uitars            419922.275966  92.331195              8.939367e+08   

                       
                 mean  
model                  
gta1    193995.080357  
qwen    197483.622689  
uitars  196556.000461

In [72]:
display(df[df['query_type'] == 'direct_query'].groupby(['model', 'reasoning_type', 'query_type', 'variant'])[['corrected_action_str_em', 'corrected_hit_box_accuracy', 'corrected_normalized_mse', 'corrected_bbox_center_mse']].agg(['sum', 'mean']))
display(df[df['query_type'] == 'direct_query'].groupby(['model', 'reasoning_type', 'query_type', 'variant'])[['action_str_em', 'hit_box_accuracy', 'bbox_center_mse']].agg(['sum', 'mean']))


corrected_action_str_em  \
                                                                   sum   
model  reasoning_type query_type   variant                               
gta1   no_reasoning   direct_query original                        298   
                                   precision                       298   
                                   style                           293   
                                   text_shrink                     295   
qwen   no_reasoning   direct_query original                        298   
                                   precision                       298   
                                   style                           293   
                                   text_shrink                     295   
       reasoning      direct_query original                        233   
                                   precision                       245   
                                   style                           234   
                                   text_shrink                     237   
uitars no_reasoning   direct_query original                        286   
                                   precision                       294   
                                   style                           283   
                                   text_shrink                     285   
       reasoning      direct_query original                        261   
                                   precision                       270   
                                   style                           251   
                                   text_shrink                     263   

                                                          \
                                                    mean   
model  reasoning_type query_type   variant                 
gta1   no_reasoning   direct_query original     1.000000   
                                   precision    1.000000   
                                   style        1.000000   
                                   text_shrink  1.000000   
qwen   no_reasoning   direct_query original     1.000000   
                                   precision    1.000000   
                                   style        1.000000   
                                   text_shrink  1.000000   
       reasoning      direct_query original     0.781879   
                                   precision    0.822148   
                                   style        0.798635   
                                   text_shrink  0.803390   
uitars no_reasoning   direct_query original     0.959732   
                                   precision    0.986577   
                                   style        0.965870   
                                   text_shrink  0.966102   
       reasoning      direct_query original     0.875839   
                                   precision    0.906040   
                                   style        0.856655   
                                   text_shrink  0.891525   

                                               corrected_hit_box_accuracy  \
                                                                      sum   
model  reasoning_type query_type   variant                                  
gta1   no_reasoning   direct_query original                           151   
                                   precision                          147   
                                   style                              146   
                                   text_shrink                        155   
qwen   no_reasoning   direct_query original                           146   
                                   precision                          144   
                                   style                              130   
                                   text_shrink                        142   
       reasoning      direct_query original                           136   
                                   prec

action_str_em            \
                                                         sum      mean   
model  reasoning_type query_type   variant                               
gta1   no_reasoning   direct_query original            246.0  0.825503   
                                   precision           247.0  0.828859   
                                   style               241.0  0.822526   
                                   text_shrink         245.0  0.830508   
qwen   no_reasoning   direct_query original            246.0  0.825503   
                                   precision           247.0  0.828859   
                                   style               241.0  0.822526   
                                   text_shrink         244.0  0.829932   
       reasoning      direct_query original             47.0  0.500000   
                                   precision            70.0  0.630631   
                                   style                45.0  0.576923   
                                   text_shrink          54.0  0.540000   
uitars no_reasoning   direct_query original            237.0  0.795302   
                                   precision           243.0  0.815436   
                                   style               233.0  0.795222   
                                   text_shrink         238.0  0.806780   
       reasoning      direct_query original            218.0  0.731544   
                                   precision           224.0  0.751678   
                                   style               212.0  0.723549   
                                   text_shrink         220.0  0.745763   

                                               hit_box_accuracy            \
                                                            sum      mean   
model  reasoning_type query_type   variant                                  
gta1   no_reasoning   direct_query original               151.0  0.506711   
                                   precision              145.0  0.486577   
                                   style                  142.0  0.484642   
                                   text_shrink            153.0  0.518644   
qwen   no_reasoning   direct_query original               144.0  0.483221   
                                   precision              139.0  0.466443   
                                   style                  125.0  0.426621   
                                   text_shrink            142.0  0.482993   
       reasoning      direct_query original                32.0  0.372093   
                                   precision               41.0  0.390476   
                                   style                   29.0  0.432836   
                                   text_shrink             38.0  0.404255   
uitars no_reasoning   direct_query original               154.0  0.516779   
                                   precision              145.0  0.486577   
                                   style                  142.0  0.484642   
                                   text_shrink            154.0  0.522034   
       reasoning      direct_query original               148.0  0.534296   
                                   precision              146.0  0.515901   
                                   style                  135.0  0.505618   
                                   text_shrink            147.0  0.525000   

                                               bbox_center_mse                 
                                                           sum           mean  
model  reasoning_type query_type   variant                                     
gta1   no_reasoning   direct_query original       3.034297e+07  101822.061852  
                                   precision      2.170189e+07   72825.143450  
                                   style          2.720100e+07   92836.170080  
                                   text_shrink    2.843740e+07   96397.964019  
qwen   no_re

# Save df as csv

In [73]:
df.to_csv('final_baseline_results.csv', index=False)

# Save eval dataset in huggingface format

In [74]:
DATA_PARENT_DIR = '/Users/lockewang/FIG/WebDomainRandomizer/training_split'
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11835 entries, 0 to 11834
Data columns (total 29 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   model                             11835 non-null  object 
 1   reasoning_type                    11835 non-null  object 
 2   query_type                        11835 non-null  object 
 3   test_split                        11835 non-null  object 
 4   variant                           11835 non-null  object 
 5   task_id                           11835 non-null  object 
 6   step_index                        11835 non-null  int64  
 7   instruction                       11835 non-null  object 
 8   gt_bbox                           11835 non-null  object 
 9   prediction                        11835 non-null  object 
 10  screenshot                        11835 non-null  object 
 11  action_str_em                     10264 non-null  float64
 12  hit_

In [75]:
pd.set_option('display.max_colwidth', None)
df['screenshot'].head(1)

0    ~/FIG/WebDomainRandomizer/test_splits/run_20251113_005227_test_task_text_shrink/8dcf6423-262a-439b-9ee7-279a920468fa/screenshots/step_1_click.png
Name: screenshot, dtype: object

In [76]:
from datasets import Dataset, Features, Image, Value, concatenate_datasets
from PIL import Image as PILImage
import json
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import gc
import sys

# Prepare data
eval_data_df = df[['variant', 'query_type', 'task_id', 'step_index', 'instruction', 'gt_bbox', 'screenshot']].copy()

# Convert gt_bbox to JSON string upfront
eval_data_df['gt_bbox'] = eval_data_df['gt_bbox'].apply(
    lambda x: json.dumps(x) if isinstance(x, (list, tuple)) else str(x) if x is not None else ''
)

# Load image function
def load_image(path):
    """Load image from file path and return PIL Image"""
    if pd.isna(path) or path == '':
        raise ValueError(f"Empty or NaN path encountered")
    path_str = str(path)
    if path_str.startswith('~'):
        path_str = path_str.replace('~', str(Path.home()))
    path_obj = Path(path_str)
    if not path_obj.exists():
        raise FileNotFoundError(f"Image file does not exist: {path_str}")
    try:
        return PILImage.open(path_obj).convert('RGB')
    except Exception as e:
        raise RuntimeError(f"Could not load image from {path_str}: {e}")

# Process row: load image and create dict entry
def process_row(original_idx, row):
    """Process a single row: load image and return dict with original index"""
    try:
        image = load_image(row['screenshot'])
        return {
            'variant': row['variant'],
            'query_type': row['query_type'],
            'task_id': row['task_id'],
            'step_index': int(row['step_index']),
            'instruction': row['instruction'],
            'gt_bbox': row['gt_bbox'],
            'screenshot': image,
            '_original_idx': original_idx,
        }, None
    except Exception as e:
        return None, (original_idx, row['screenshot'], str(e))

# Process in batches to prevent memory issues
BATCH_SIZE = 500  # Process 500 images at a time
MAX_WORKERS = 4   # Reduced workers to prevent memory overload

features = Features({
    'variant': Value('string'),
    'query_type': Value('string'),
    'task_id': Value('string'),
    'step_index': Value('int32'),
    'instruction': Value('string'),
    'gt_bbox': Value('string'),
    'screenshot': Image(),
})

print(f"Processing {len(eval_data_df)} examples in batches of {BATCH_SIZE}...")
datasets = []
failed_paths = []

# Process in batches
for batch_start in range(0, len(eval_data_df), BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, len(eval_data_df))
    batch_df = eval_data_df.iloc[batch_start:batch_end]
    
    print(f"Processing batch {batch_start//BATCH_SIZE + 1}/{(len(eval_data_df)-1)//BATCH_SIZE + 1} (rows {batch_start}-{batch_end-1})...")
    
    data_list = []
    
    # Process batch with parallel loading
    try:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {
                executor.submit(process_row, original_idx, row): original_idx
                for original_idx, row in batch_df.iterrows()
            }
            
            for future in tqdm(as_completed(futures), total=len(futures), desc=f"Batch {batch_start//BATCH_SIZE + 1}", leave=False):
                result, error = future.result()
                if error:
                    failed_paths.append(error)
                else:
                    data_list.append(result)
        
        # Check for failures in this batch
        if failed_paths and len([f for f in failed_paths if batch_start <= f[0] < batch_end]) > 0:
            batch_failures = [f for f in failed_paths if batch_start <= f[0] < batch_end]
            print(f"  Warning: {len(batch_failures)} failures in this batch")
        
        # Sort by original index
        data_list = sorted(data_list, key=lambda x: x['_original_idx'])
        for item in data_list:
            del item['_original_idx']
        
        # Create dataset for this batch
        batch_dataset = Dataset.from_list(data_list, features=features)
        datasets.append(batch_dataset)
        
        # Free memory
        del data_list
        gc.collect()
        
    except MemoryError:
        print(f"\nERROR: Out of memory processing batch {batch_start//BATCH_SIZE + 1}")
        print("Try reducing BATCH_SIZE or MAX_WORKERS")
        raise
    except Exception as e:
        print(f"\nERROR: Failed to process batch {batch_start//BATCH_SIZE + 1}: {e}")
        raise

# Check for failures
if failed_paths:
    print(f"\nERROR: Failed to load {len(failed_paths)} images:")
    for idx, path, error in failed_paths[:10]:
        print(f"  Row {idx}: {path} - {error}")
    if len(failed_paths) > 10:
        print(f"  ... and {len(failed_paths) - 10} more failures")
    raise RuntimeError(f"Failed to load {len(failed_paths)} images. Please check the file paths.")

# Concatenate all batch datasets
print("Combining batches...")
eval_dataset = concatenate_datasets(datasets)
print(f"✓ Dataset created with {len(eval_dataset)} examples")

# Free memory
del datasets
gc.collect()

# Save dataset
print("Saving dataset to disk...")
try:
    eval_dataset.save_to_disk("eval_dataset")
    print("✓ Dataset saved to eval_dataset/ directory")
except Exception as e:
    print(f"ERROR saving dataset: {e}")
    raise

/Users/lockewang/FIG/WebDomainRandomizer/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Processing 11835 examples in batches of 500...
Processing batch 1/24 (rows 0-499)...


Processing batch 2/24 (rows 500-999)...


Processing batch 3/24 (rows 1000-1499)...


Processing batch 4/24 (rows 1500-1999)...


...

Processing batch 23/24 (rows 11000-11499)...


Processing batch 24/24 (rows 11500-11834)...


Combining batches...
✓ Dataset created with 11835 examples
Saving dataset to disk...


Saving the dataset (12/12 shards): 100%|██████████| 11835/11835 [00:05<00:00, 2142.41 examples/s]

✓ Dataset saved to eval_dataset/ directory


In [77]:
from datasets import load_from_disk

# Load dataset from disk (properly handles Image features)
eval_dataset = load_from_disk("eval_dataset")
print(f"Loaded dataset with {len(eval_dataset)} examples")

Loaded dataset with 11835 examples


In [ ]:
print(eval_dataset)
print(eval_dataset.features)

In [ ]:
example = eval_dataset[0]